# Apple Intelligence and HarmonyOS usability: data collection, classification, and analysis

Pipeline for the Apple/HarmonyOS part of the usability evaluation in "When Convenience Meets Risk: Evaluating
Usability, Security, and AI Reliability Across Modern Operating Systems" (Alhawarat and Nijem).

**Steps**
1. Collect public posts through official APIs (YouTube, Hacker News, Bluesky, Mastodon, Lemmy, App Store),
   breadth-first: at most 40 posts per cluster (video, thread, author, or app), because posts responding to the
   same stimulus are not independent.
2. Classify AI-feature relevance and extract aspect-sentiment pairs with Claude Sonnet 5 (Section 5), then
   re-check relevance with a stricter verification pass (Section 5.3).
3. Audit relevance with transparent rules and validate the pipeline against two independent human raters.
4. Report the positive share with cluster-bootstrap confidence intervals and a reportability rule (Section 6),
   and derive themes by clustering (Sections 7-8).

**Caching.** Every model result is cached on disk, keyed to a fingerprint of the exact prompt, model, and settings
(`PROMPT_VERSION`, `601ace4682` for the extraction step). Re-running the notebook reuses cached results and only
classifies posts that have no cached result under the same fingerprint. Collection state is also saved, so
interrupted collection resumes without refetching.

**Credentials (Colab Secrets).** `ANTHROPIC_API_KEY` (classification), `YOUTUBE_API_KEY` (YouTube), and optionally
`BLUESKY_HANDLE` and `BLUESKY_APP_PASSWORD`. Paid steps are gated: `NO_NEW_SPEND = True` disables all API calls
that incur usage charges, and each paid step has a budget cap.

**Dataset versions.** File names carry the collection round: v2 (initial multi-source sample), v3 (breadth-first
re-sampling), v4 (expanded collection with a stricter pre-classification filter). Earlier files are never overwritten.
All figures are saved at 300 dpi.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = "/content/drive/My Drive/OS_Trends_NFR/UsbilityData"
OUT_DIR  = "/content/drive/My Drive/OS_Trends_NFR/UsbilityData/forum_collection_output"

import os
os.makedirs(OUT_DIR, exist_ok=True)
print("Output dir:", OUT_DIR)


In [ ]:
!pip install -q anthropic sentence-transformers beautifulsoup4 requests


In [ ]:
import time
import re
import json
import urllib.robotparser
from urllib.parse import urlparse

import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np

import torch
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import CountVectorizer
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from scipy.spatial.distance import squareform
import matplotlib.pyplot as plt

import anthropic
from google.colab import userdata

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

client = anthropic.Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))
print("Anthropic client ready.")


## 2. Collection design: official public APIs only, breadth-first

| Source | Platform(s) | Credentials | Cluster (independence unit) |
|---|---|---|---|
| YouTube Data API v3 | Apple + HarmonyOS | `YOUTUBE_API_KEY` | video |
| Hacker News (Algolia) | Apple + HarmonyOS | none | story/thread |
| Bluesky (AT Protocol) | Apple + HarmonyOS | `BLUESKY_HANDLE` (full handle, e.g. `name.bsky.social`) + `BLUESKY_APP_PASSWORD` | author |
| Mastodon (public hashtag timelines) | Apple + HarmonyOS | none | author |
| Lemmy (search API) | Apple + HarmonyOS | none | author |
| App Store reviews RSS | Apple | none | app |

The YouTube Data API is quota-based (a fixed daily allowance) and does not bill per call. Sources that disallow
automated access or present bot-verification challenges are excluded. Without a YouTube key, the notebook uses
only previously fetched YouTube comments.

In [ ]:
import html as html_lib
import os
import hashlib

FORCE_RECOLLECT = False
RAW_POSTS_V3 = f"{OUT_DIR}/raw_collected_posts_v3.csv"
RAW_POSTS_PATH = f"{OUT_DIR}/raw_collected_posts_v4.csv"
import shutil
if not os.path.exists(RAW_POSTS_PATH) and os.path.exists(RAW_POSTS_V3):
    shutil.copy(RAW_POSTS_V3, RAW_POSTS_PATH)
    print("v4 dataset seeded from", RAW_POSTS_V3)
SKIP_COLLECTION = os.path.exists(RAW_POSTS_PATH) and not FORCE_RECOLLECT
print("SKIP_COLLECTION =", SKIP_COLLECTION,
      "(reusing saved posts)" if SKIP_COLLECTION else "(will collect from the APIs)")

TARGET_POSTS_PER_PLATFORM = 10**9
MAX_POSTS_PER_CLUSTER = 40

AI_VOCAB = (r"\bai\b|a\.i\.|artificial intelligence|intelligen|assistant|\bagents?\b|chatbot|\bllm|"
            r"summar|translat|voice|generat|\bgpt|gemini|deepseek|copilot|"
            r"apple intelligence|writing tools?|genmoji|image playground|\bsiri\b|clean ?up|"
            r"private cloud compute|harmony intelligence|celia|xiao ?yi|pangu")
AI_PREFILTER = re.compile(AI_VOCAB, re.IGNORECASE)
PLATFORM_TERMS = {
    "apple": re.compile(r"apple|iphone|ipad|\bmac|ios|siri|genmoji|image playground|writing tools?",
                        re.IGNORECASE),
    "harmonyos": re.compile(r"harmony ?os|huawei|celia|xiao ?yi|hongmeng|mate ?\d|pura", re.IGNORECASE),
}

def passes_filter(text, platform, ai_specific, require_platform_term=False):
    if require_platform_term and not PLATFORM_TERMS[platform].search(text):
        return False
    return True if ai_specific else bool(AI_PREFILTER.search(text))

def mostly_english(text, threshold=0.9):
    letters = [ch for ch in text if ch.isalpha()]
    if not letters:
        return False
    return sum(1 for ch in letters if ch.isascii()) / len(letters) >= threshold

MIN_TEXT_LENGTH = 20
REQUEST_DELAY = 0.5

def get_json(url, params=None, headers=None, timeout=20):
    try:
        r = requests.get(url, params=params, headers=headers, timeout=timeout)
        time.sleep(REQUEST_DELAY)
        if r.status_code != 200:
            return None, f"HTTP {r.status_code}: {r.text[:200]}"
        return r.json(), None
    except Exception as e:
        return None, str(e)

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None
YOUTUBE_API_KEY = secret('YOUTUBE_API_KEY')
BLUESKY_HANDLE = secret('BLUESKY_HANDLE')
BLUESKY_APP_PASSWORD = secret('BLUESKY_APP_PASSWORD')
print("YouTube key:", bool(YOUTUBE_API_KEY), "| Bluesky credentials:", bool(BLUESKY_HANDLE and BLUESKY_APP_PASSWORD))

collection_log = []

NO_NEW_SPEND = True


## 2.1 Classifier definition (no API calls)

Defines the extraction prompt, model (Claude Sonnet 5, extended thinking disabled), output parsing, and the
on-disk cache. The prompt, model, `thinking` setting, and `max_tokens` together determine `PROMPT_VERSION`;
cached results are reused only when it matches. Editing this cell changes the fingerprint, so all posts would be
re-classified. The next cell reports whether the cached results match.

In [ ]:
EXTRACTION_SYSTEM_PROMPT = '''You are classifying user-generated text (app-store reviews, video comments, or
discussion-forum/social posts) about operating-system AI features:
- Apple: Apple Intelligence features (Writing Tools, Clean Up, Genmoji, Image Playground,
  notification/email summaries, Siri with Apple Intelligence).
- HarmonyOS: Huawei's AI features (Celia/XiaoYi assistant and agents, Harmony Intelligence,
  other AI-driven HarmonyOS features).

Task:
1. Determine whether the post discusses the platform's AI feature(s) positively or negatively.
   Simply mentioning the feature exists, or discussing something else entirely (e.g. the OS in
   general, geopolitics, hardware, other companies' products), is NOT enough -- the post must
   express an actual opinion, issue, or positive attribute of the platform's AI feature.
   The context line (e.g. the video title) tells you what the post is responding to.
2. If yes, produce a dictionary of short summaries (max 3 words each), each mapped to a
   sentiment of "positive", "negative", or "neutral". Summaries must be concise and specific
   to the AI feature (ignore unrelated topics).
3. If no genuine AI-feature opinion is present, return an empty summaries dictionary.

Return ONLY the JSON object below. No explanation, no commentary, no text before or after it:
{"ai_related": "Yes" or "No", "summaries": {"<short_summary>": "<positive/negative/neutral>"}}
'''

EXTRACTION_MODEL = "claude-sonnet-5"
PRICES_PER_MTOK = {"claude-sonnet-5": (2.00, 10.00),
                   "claude-haiku-4-5-20251001": (1.00, 5.00)}
BATCH_DISCOUNT = 0.5
MAX_TOKENS = 1000
THINKING = {"type": "disabled"}
VALID_SENTIMENTS = {"positive", "negative", "neutral"}
PROMPT_VERSION = hashlib.sha1((EXTRACTION_SYSTEM_PROMPT + EXTRACTION_MODEL + json.dumps(THINKING, sort_keys=True)
                               + str(MAX_TOKENS)).encode()).hexdigest()[:10]
CACHE_PATH = f"{OUT_DIR}/extraction_cache.jsonl"
print("Model:", EXTRACTION_MODEL, "| PROMPT_VERSION:", PROMPT_VERSION)

def post_key(row):
    return f"{PROMPT_VERSION}|{row['platform']}|{row['url']}"

def build_user_msg(row):
    ctx = row.get("context", "")
    ctx = "" if (ctx is None or (isinstance(ctx, float) and pd.isna(ctx))) else str(ctx)
    return (f"Platform: {row['platform']}. Context: {ctx or row['feature_tag']}. "
            f"Classify the following user post:\n\n{str(row['text'])[:2000]}")

def parse_json_response(raw):
    text = re.sub(r"```(?:json)?", "", raw)
    start = text.find("{")
    if start == -1:
        raise ValueError("no JSON object in response")
    obj, _ = json.JSONDecoder().raw_decode(text[start:])
    if not isinstance(obj, dict):
        raise ValueError("top-level JSON is not an object")
    summ = obj.get("summaries") or {}
    summ = summ if isinstance(summ, dict) else {}
    obj["summaries"] = {str(k).strip(): str(v).strip().lower() for k, v in summ.items()
                        if str(k).strip() and str(v).strip().lower() in VALID_SENTIMENTS}
    obj["ai_related"] = "Yes" if str(obj.get("ai_related", "")).strip().lower() == "yes" else "No"
    return obj

def load_cache():
    cache = {}
    if os.path.exists(CACHE_PATH):
        with open(CACHE_PATH, encoding="utf-8") as f:
            for line in f:
                try:
                    rec = json.loads(line)
                    cache[rec["key"]] = rec
                except Exception:
                    continue
    return cache

def append_cache(key, result, in_tok, out_tok, mode):
    rec = {"key": key, "result": result, "in_tok": in_tok, "out_tok": out_tok, "mode": mode}
    with open(CACHE_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(rec) + "\n")
    return rec

def response_text(msg):
    return "".join(getattr(b, "text", "") for b in (msg.content or []) if getattr(b, "type", "") == "text")

def check_response(msg):
    if getattr(msg, "stop_reason", None) == "max_tokens":
        return "truncated: hit max_tokens before the answer finished"
    return None

def extract_sync(row):
    try:
        resp = client.messages.create(model=EXTRACTION_MODEL, max_tokens=MAX_TOKENS, thinking=THINKING,
                                      system=EXTRACTION_SYSTEM_PROMPT,
                                      messages=[{"role": "user", "content": build_user_msg(row)}])
    except Exception as e:
        return None, 0, 0, f"api: {e}"
    it, ot = resp.usage.input_tokens, resp.usage.output_tokens
    bad = check_response(resp)
    if bad:
        return None, it, ot, bad
    raw = response_text(resp)
    try:
        return parse_json_response(raw), it, ot, None
    except Exception as e:
        return None, it, ot, f"parse: {e} | raw: {raw[:200]}"

def cost_usd(in_tok, out_tok, batch):
    p_in, p_out = PRICES_PER_MTOK[EXTRACTION_MODEL]
    c = (in_tok * p_in + out_tok * p_out) / 1e6
    return c * (BATCH_DISCOUNT if batch else 1.0)


In [ ]:
_c = load_cache()
_versions = __import__("collections").Counter(k.split("|", 1)[0] for k in _c)
print(f"PROMPT_VERSION now: {PROMPT_VERSION} | cached results by version: {dict(_versions)}")
if PROMPT_VERSION == "601ace4682":
    print(f"OK: {_versions.get(PROMPT_VERSION, 0)} cached results from earlier runs will be reused.")
else:
    print("WARNING: PROMPT_VERSION differs from the reference run (601ace4682). Previously classified posts would be "
          "re-classified. Restore Section 2b to the original before continuing.")


## 3.1 Apple App Store reviews

In [ ]:
if SKIP_COLLECTION:
    print('Skipped: using saved posts (set FORCE_RECOLLECT = True in Section 2 to recollect).')
    appstore_posts = []
else:
    APP_STORE_COUNTRIES = ["us", "gb", "ca", "au", "in", "ie", "nz", "sg", "za", "ph", "my"]
    APPLE_APP_TERMS = ["Image Playground", "Pages", "Keynote", "Numbers", "Notes", "Mail", "Freeform", "Journal"]

    apple_apps = {}
    for term in APPLE_APP_TERMS:
        for entity in ["software", "macSoftware"]:
            data, err = get_json("https://itunes.apple.com/search",
                                 params={"term": term, "entity": entity, "country": "us", "limit": 10})
            if err or not data:
                collection_log.append(("app_store", "apple", f"search '{term}' ({entity}) failed: {err}"))
                continue
            for res in data.get("results", []):
                if res.get("artistName") == "Apple" and term.lower() in res.get("trackName", "").lower():
                    apple_apps[res["trackId"]] = res["trackName"]
    print(f"Resolved {len(apple_apps)} Apple first-party app listings")

    appstore_posts, seen_review_ids = [], set()
    for app_id, app_name in apple_apps.items():
        n = 0
        for cc in APP_STORE_COUNTRIES:
            for page in range(1, 11):
                url = f"https://itunes.apple.com/{cc}/rss/customerreviews/page={page}/id={app_id}/sortby=mostrecent/json"
                data, err = get_json(url)
                entries = (data or {}).get("feed", {}).get("entry", [])
                if isinstance(entries, dict):
                    entries = [entries]
                if err or not entries:
                    break
                for e in entries:
                    if "im:rating" not in e:
                        continue
                    rid = e.get("id", {}).get("label", "")
                    text = f"{e.get('title', {}).get('label', '')}. {e.get('content', {}).get('label', '')}".strip()
                    if rid in seen_review_ids or len(text) < MIN_TEXT_LENGTH:
                        continue
                    if not passes_filter(text, "apple", ai_specific=(app_name == "Image Playground")):
                        continue
                    seen_review_ids.add(rid)
                    appstore_posts.append({"url": f"{url}#review={rid}", "platform": "apple",
                                           "feature_tag": "appstore_" + re.sub(r"\W+", "_", app_name.lower()),
                                           "username": e.get("author", {}).get("name", {}).get("label", "unknown"),
                                           "text": text, "context": f"App Store review of Apple's app '{app_name}'",
                                           "cluster": "app:" + app_name})
                    n += 1
        collection_log.append(("app_store", "apple", f"{app_name}: {n} reviews"))
    print(f"App Store total: {len(appstore_posts)}")


## 3.2 YouTube comments (breadth-first, resumable across quota resets)

In [ ]:
if SKIP_COLLECTION:
    print('Skipped: using saved posts (set FORCE_RECOLLECT = True in Section 2 to recollect).')
    youtube_posts = []
else:
    YT = "https://www.googleapis.com/youtube/v3"
    YT_QUERIES = {
        "apple": [
            ("Apple Intelligence review", True), ("Apple Intelligence worth it", True),
            ("Apple Intelligence writing tools", True), ("Apple Intelligence clean up photos", True),
            ("Genmoji", True), ("Image Playground review", True),
            ("Apple Intelligence notification summaries", True), ("Siri Apple Intelligence", True),
            ("Apple Intelligence vs Galaxy AI", True), ("Apple Intelligence disappointing", True),
            ("iOS 18 review", False), ("iOS 26 review", False),
        ],
        "harmonyos": [
            ("HarmonyOS AI features", True), ("Huawei Celia AI assistant", True),
            ("Xiaoyi AI agent HarmonyOS", True), ("Harmony Intelligence Huawei", True),
            ("HarmonyOS 6 AI agent", True),
            ("HarmonyOS NEXT review", False), ("HarmonyOS 5 review", False), ("HarmonyOS 6 review", False),
            ("HarmonyOS 5 long term", False), ("Huawei Mate 70 HarmonyOS", False),
            ("Huawei Pura 80 HarmonyOS", False), ("Huawei Mate X6 HarmonyOS", False),
        ],
    }
    YT_SEARCH_PAGES = 2
    YT_MAX_COMMENTS_PER_VIDEO = 40
    YT_MAX_PAGES_PER_VIDEO = 3
    YT_UNIT_BUDGET = 9000
    YT_TARGET_PER_PLATFORM = 4500
    YT_PUBLISHED_AFTER = "2024-06-01T00:00:00Z"
    YT_STATE_PATH = f"{OUT_DIR}/youtube_state_v3.json"
    YT_PREV_STATE_PATH = f"{OUT_DIR}/youtube_state.json"

    class YTQuotaStop(Exception):
        pass

    def yt_get(endpoint, params):
        params = dict(params, key=YOUTUBE_API_KEY)
        try:
            r = requests.get(f"{YT}/{endpoint}", params=params, timeout=20)
            time.sleep(REQUEST_DELAY)
        except Exception as e:
            return None, f"network: {e}"
        if r.status_code == 200:
            return r.json(), None
        try:
            reason = r.json()["error"]["errors"][0]["reason"]
        except Exception:
            reason = f"HTTP {r.status_code}"
        if reason in ("quotaExceeded", "dailyLimitExceeded", "rateLimitExceeded"):
            raise YTQuotaStop(reason)
        return None, reason

    def slug(q):
        return re.sub(r"\W+", "_", q.lower()).strip("_")

    youtube_posts = []
    if True:
        if os.path.exists(YT_STATE_PATH):
            state = json.load(open(YT_STATE_PATH))
            print(f"Resuming v3 YouTube state: {len(state['videos'])} videos, {len(state['done'])} done, "
                  f"{len(state['comments'])} comments so far")
        else:
            state = {"videos": {}, "searched": [], "done": [], "comments": []}
            if os.path.exists(YT_PREV_STATE_PATH):
                prev = json.load(open(YT_PREV_STATE_PATH))
                for k, v in prev.get("videos", {}).items():
                    q = v.get("query")
                    state["videos"][k] = {"platform": v["platform"], "vid": v["vid"], "queries": [q] if q else [],
                                          "ai_specific": bool(v.get("ai_specific")), "title": v.get("title", ""),
                                          **({"comment_count": v["comment_count"]} if "comment_count" in v else {})}
                state["searched"] = list(prev.get("searched", []))
                for c in prev.get("comments", []):
                    c = dict(c)
                    c["cluster"] = "yt:" + re.search(r"v=([^&]+)", c["url"]).group(1)
                    state["comments"].append(c)
                state["done"] = list(prev.get("done", []))
                print(f"Reusing {len(state['videos'])} videos found by the earlier collection (no repeated searches) and importing "
                      f"its {len(state['comments'])} already-fetched comments from {len(state['done'])} videos.")
        save_state = lambda: json.dump(state, open(YT_STATE_PATH, "w"))
        save_state()
        units = 0
        if not YOUTUBE_API_KEY:
            print("No YOUTUBE_API_KEY -- using only the imported v2 comments (no new videos). Add a key for breadth.")
            collection_log.append(("youtube", "both", "no API key -- v2 comments only"))
        else:
          try:
              for platform, queries in YT_QUERIES.items():
                  for query, ai_specific in queries:
                      if f"{platform}|{query}" in state["searched"]:
                          continue
                      token = None
                      for _ in range(YT_SEARCH_PAGES):
                          p = {"part": "snippet", "q": query, "type": "video", "maxResults": 50,
                               "relevanceLanguage": "en", "order": "relevance", "publishedAfter": YT_PUBLISHED_AFTER}
                          if token:
                              p["pageToken"] = token
                          data, err = yt_get("search", p)
                          if err or not data:
                              collection_log.append(("youtube", platform, f"search '{query}': {err}"))
                              break
                          for it in data.get("items", []):
                              vid = it.get("id", {}).get("videoId")
                              if not vid:
                                  continue
                              k = f"{platform}|{vid}"
                              v = state["videos"].setdefault(k, {"platform": platform, "vid": vid, "queries": [],
                                                                 "ai_specific": False,
                                                                 "title": html_lib.unescape(it["snippet"].get("title", ""))})
                              if query not in v["queries"]:
                                  v["queries"].append(query)
                              v["ai_specific"] = v["ai_specific"] or ai_specific
                          token = data.get("nextPageToken")
                          if not token:
                              break
                      state["searched"].append(f"{platform}|{query}")
                      save_state()
              print(f"Videos known: {len(state['videos'])}")

              need = sorted({v["vid"] for v in state["videos"].values() if "comment_count" not in v})
              for i in range(0, len(need), 50):
                  chunk = need[i:i + 50]
                  data, err = yt_get("videos", {"part": "statistics", "id": ",".join(chunk)})
                  units += 1
                  counts = {it["id"]: int(it.get("statistics", {}).get("commentCount", 0)) for it in (data or {}).get("items", [])}
                  for v in state["videos"].values():
                      if v["vid"] in chunk:
                          v["comment_count"] = counts.get(v["vid"], 0)
              save_state()

              done = set(state["done"])
              seen = {(c["platform"], c["url"]) for c in state["comments"]}
              for platform in YT_QUERIES:
                  _per_vid = __import__("collections").Counter(c["cluster"] for c in state["comments"] if c["platform"] == platform)
                  have = sum(min(n, YT_MAX_COMMENTS_PER_VIDEO) for n in _per_vid.values())
                  vids = sorted((k for k, v in state["videos"].items() if v["platform"] == platform),
                                key=lambda k: (not state["videos"][k]["ai_specific"], -state["videos"][k].get("comment_count", 0)))
                  for k in vids:
                      if have >= YT_TARGET_PER_PLATFORM:
                          break
                      if k in done:
                          continue
                      v = state["videos"][k]
                      if v.get("comment_count", 1) == 0:
                          done.add(k)
                          continue
                      kept_here, token = 0, None
                      for _ in range(YT_MAX_PAGES_PER_VIDEO):
                          if units >= YT_UNIT_BUDGET:
                              raise YTQuotaStop("local unit budget reached")
                          p = {"part": "snippet,replies", "videoId": v["vid"], "maxResults": 100,
                               "textFormat": "plainText", "order": "relevance"}
                          if token:
                              p["pageToken"] = token
                          data, err = yt_get("commentThreads", p)
                          units += 1
                          if err or not data:
                              break
                          for item in data.get("items", []):
                              top = item["snippet"]["topLevelComment"]
                              for c in [top] + item.get("replies", {}).get("comments", []):
                                  if kept_here >= YT_MAX_COMMENTS_PER_VIDEO:
                                      break
                                  text = c["snippet"].get("textOriginal", "").strip()
                                  url = f"https://www.youtube.com/watch?v={v['vid']}&lc={c['id']}"
                                  if (platform, url) in seen or len(text) < MIN_TEXT_LENGTH or not mostly_english(text):
                                      continue
                                  if not passes_filter(text, platform, v["ai_specific"]):
                                      continue
                                  seen.add((platform, url))
                                  state["comments"].append({
                                      "url": url, "platform": platform,
                                      "feature_tag": "yt_" + slug(v["queries"][0] if v["queries"] else "unknown"),
                                      "username": c["snippet"].get("authorDisplayName", "unknown"),
                                      "text": text, "context": f"Comment on YouTube video: {v['title']}",
                                      "cluster": "yt:" + v["vid"]})
                                  kept_here += 1
                                  have += 1
                          token = data.get("nextPageToken")
                          if kept_here >= YT_MAX_COMMENTS_PER_VIDEO or not token:
                              break
                      done.add(k)
                      state["done"] = sorted(done)
                      save_state()
                  print(f"  [{platform}] YouTube comments kept: {have} from "
                        f"{len({c['cluster'] for c in state['comments'] if c['platform'] == platform})} videos")
          except YTQuotaStop as e:
              save_state()
              print(f"YouTube stopped early ({e}). Progress saved -- re-run this cell after the daily quota "
                    f"resets (midnight Pacific); nothing already fetched is repeated.")
              collection_log.append(("youtube", "both", f"stopped early: {e} (resumable)"))
        youtube_posts = state["comments"]
        for platform in YT_QUERIES:
            collection_log.append(("youtube", platform,
                                   f"{sum(1 for c in youtube_posts if c['platform'] == platform)} comments (units this run: {units})"))
    print(f"YouTube total: {len(youtube_posts)}")


## 3.3 Hacker News comments

In [ ]:
if SKIP_COLLECTION:
    print('Skipped: using saved posts (set FORCE_RECOLLECT = True in Section 2 to recollect).')
    hn_posts = []
else:
    HN_QUERIES = {
        "apple": [("Apple Intelligence", True), ("Genmoji", True), ("Image Playground", True),
                  ("Writing Tools", True), ("Private Cloud Compute", True),
                  ("notification summaries", True), ("Siri Apple Intelligence", True), ("Apple AI", False)],
        "harmonyos": [("HarmonyOS", False), ("Harmony OS", False), ("HarmonyOS NEXT", False),
                      ("Celia Huawei", True), ("Xiaoyi Huawei", True), ("Huawei AI assistant", True)],
    }
    HN_SINCE_TIMESTAMP = 1717200000
    HN_MAX_PAGES = 10

    hn_posts, seen_hn = [], set()
    for platform, queries in HN_QUERIES.items():
        for query, ai_specific in queries:
            n = 0
            for page in range(HN_MAX_PAGES):
                data, err = get_json("https://hn.algolia.com/api/v1/search", params={
                    "query": query, "tags": "comment", "hitsPerPage": 100, "page": page,
                    "numericFilters": f"created_at_i>{HN_SINCE_TIMESTAMP}"})
                if err or not data:
                    collection_log.append(("hacker_news", platform, f"'{query}' p{page}: {err}"))
                    break
                for h in data.get("hits", []):
                    oid = h.get("objectID")
                    text = BeautifulSoup(html_lib.unescape(h.get("comment_text") or ""), "html.parser").get_text(" ", strip=True)
                    if (platform, oid) in seen_hn or len(text) < MIN_TEXT_LENGTH:
                        continue
                    if not passes_filter(text, platform, ai_specific, require_platform_term=True):
                        continue
                    seen_hn.add((platform, oid))
                    hn_posts.append({"url": f"https://news.ycombinator.com/item?id={oid}", "platform": platform,
                                     "feature_tag": "hn_" + re.sub(r"\W+", "_", query.lower()),
                                     "username": h.get("author", "unknown"), "text": text,
                                     "context": f"Hacker News comment on the story: {h.get('story_title') or ''}",
                                     "cluster": f"hn:{h.get('story_id') or oid}"})
                    n += 1
                if page + 1 >= data.get("nbPages", 0):
                    break
            collection_log.append(("hacker_news", platform, f"'{query}': {n}"))
            print(f"  [{platform}] HN '{query}': {n}")
    print(f"Hacker News total: {len(hn_posts)}")


## 3.4 Bluesky posts

In [ ]:
if SKIP_COLLECTION:
    print('Skipped: using saved posts (set FORCE_RECOLLECT = True in Section 2 to recollect).')
    bsky_posts = []
else:
    BSKY_PDS = "https://bsky.social"
    BSKY_QUERIES = {
        "apple": [("Apple Intelligence", True), ("Genmoji", True), ("Image Playground", True),
                  ("Apple writing tools", True), ("Siri Apple Intelligence", True),
                  ("notification summaries", True)],
        "harmonyos": [("HarmonyOS", False), ("Harmony OS", False), ("Huawei Celia", True),
                      ("HarmonyOS AI", True), ("Huawei Xiaoyi", True)],
    }
    BSKY_MAX_PAGES = 10
    BSKY_SINCE = "2024-06-01T00:00:00Z"

    bsky_posts = []
    if not (BLUESKY_HANDLE and BLUESKY_APP_PASSWORD):
        print("Skipping Bluesky: add BLUESKY_HANDLE and BLUESKY_APP_PASSWORD to Colab Secrets to enable.")
        collection_log.append(("bluesky", "both", "skipped -- no credentials"))
    else:
        _handle = BLUESKY_HANDLE.strip().lstrip("@").strip()
        _pw = BLUESKY_APP_PASSWORD.strip()
        if _handle != BLUESKY_HANDLE or _pw != BLUESKY_APP_PASSWORD:
            print("Note: removed surrounding whitespace / a leading '@' from the Bluesky secrets.")
        try:
            r = requests.post(f"{BSKY_PDS}/xrpc/com.atproto.server.createSession",
                              json={"identifier": _handle, "password": _pw}, timeout=20)
            if r.status_code != 200:
                try:
                    err = r.json()
                except Exception:
                    err = {"error": f"HTTP {r.status_code}", "message": r.text[:200]}
                print(f"Bluesky rejected the login: HTTP {r.status_code} | error: {err.get('error')} | "
                      f"message: {err.get('message')}")
                print(f"  handle used: '{_handle}' | app password length: {len(_pw)} "
                      f"(app passwords look like xxxx-xxxx-xxxx-xxxx = 19 characters)")
            r.raise_for_status()
            auth = {"Authorization": f"Bearer {r.json()['accessJwt']}"}
        except Exception as e:
            auth = None
            print(f"Bluesky login failed ({e}) -- skipping. BLUESKY_HANDLE must be the FULL handle, e.g. yourname.bsky.social (no @); BLUESKY_APP_PASSWORD must be an app password, not your login password.")
            collection_log.append(("bluesky", "both", f"login failed: {e}"))
        if auth:
            seen_bsky = set()
            for platform, queries in BSKY_QUERIES.items():
                for query, ai_specific in queries:
                    n, cursor = 0, None
                    for _ in range(BSKY_MAX_PAGES):
                        p = {"q": query, "limit": 100, "lang": "en", "since": BSKY_SINCE}
                        if cursor:
                            p["cursor"] = cursor
                        data, err = get_json(f"{BSKY_PDS}/xrpc/app.bsky.feed.searchPosts", params=p, headers=auth)
                        if err or not data:
                            collection_log.append(("bluesky", platform, f"'{query}': {err}"))
                            break
                        for post in data.get("posts", []):
                            uri = post.get("uri", "")
                            text = (post.get("record", {}) or {}).get("text", "").strip()
                            if (platform, uri) in seen_bsky or len(text) < MIN_TEXT_LENGTH:
                                continue
                            if not passes_filter(text, platform, ai_specific, require_platform_term=True):
                                continue
                            seen_bsky.add((platform, uri))
                            handle = post.get("author", {}).get("handle", "unknown")
                            bsky_posts.append({"url": f"https://bsky.app/profile/{handle}/post/{uri.split('/')[-1]}",
                                               "platform": platform,
                                               "feature_tag": "bsky_" + re.sub(r"\W+", "_", query.lower()),
                                               "username": handle, "text": text, "context": "Bluesky post", "cluster": "bsky:" + handle})
                            n += 1
                        cursor = data.get("cursor")
                        if not cursor or not data.get("posts"):
                            break
                    collection_log.append(("bluesky", platform, f"'{query}': {n}"))
                    print(f"  [{platform}] Bluesky '{query}': {n}")
    print(f"Bluesky total: {len(bsky_posts)}")


## 4. Combine sources: per-cluster cap, per-platform cap, classification preview

In [ ]:
if SKIP_COLLECTION:
    posts_df = pd.read_csv(RAW_POSTS_PATH)
    posts_df['context'] = posts_df['context'].fillna('')
    print(f'Loaded {len(posts_df)} saved posts (not overwritten).')
    print(posts_df.groupby('platform').agg(posts=('url', 'size'), clusters=('cluster', 'nunique')).to_string())
else:
    all_posts = appstore_posts + youtube_posts + hn_posts + bsky_posts
    cols = ["url", "platform", "feature_tag", "username", "text", "context", "cluster"]
    posts_df = pd.DataFrame(all_posts, columns=cols)
    posts_df = posts_df.drop_duplicates(subset=["platform", "url"]).drop_duplicates(subset=["platform", "text"]).reset_index(drop=True)
    posts_df["source"] = posts_df["url"].str.extract(r"https?://(?:www\.)?([^/]+)")[0]
    _cache = load_cache()
    posts_df["_cached"] = [post_key(r) in _cache for _, r in posts_df.iterrows()]
    print("Collected (before caps), by platform and source:")
    print(posts_df.groupby(["platform", "source"]).agg(posts=("url", "size"), already_paid=("_cached", "sum")).to_string())

    def prefer_cached(g, n, seed=42):
        if len(g) <= n:
            return g
        c = g[g["_cached"]].sample(frac=1, random_state=seed)
        u = g[~g["_cached"]].sample(frac=1, random_state=seed)
        return pd.concat([c, u]).head(n)

    before = len(posts_df)
    posts_df = pd.concat([prefer_cached(g, MAX_POSTS_PER_CLUSTER) for _, g in posts_df.groupby(["platform", "cluster"])],
                         ignore_index=True)
    print(f"\nPer-cluster cap ({MAX_POSTS_PER_CLUSTER}): {before} -> {len(posts_df)} posts")

    capped = []
    for platform, grp in posts_df.groupby("platform"):
        if len(grp) > TARGET_POSTS_PER_PLATFORM:
            frac = TARGET_POSTS_PER_PLATFORM / len(grp)
            grp = pd.concat([prefer_cached(g, int(round(frac * len(g)))) for _, g in grp.groupby("source")])
            print(f"{platform}: capped to {len(grp)} posts (proportional by source, already-classified first)")
        capped.append(grp)
    posts_df = pd.concat(capped, ignore_index=True) if capped else posts_df

    print(f"\nTotal posts for extraction: {len(posts_df)}")
    summary = posts_df.groupby(["platform", "source"]).agg(posts=("url", "size"), clusters=("cluster", "nunique"),
                                                           already_paid=("_cached", "sum"))
    summary["new_to_classify"] = summary["posts"] - summary["already_paid"]
    print(summary.to_string())
    n_new = int((~posts_df["_cached"]).sum())
    print(f"\nCLASSIFICATION PREVIEW: {int(posts_df['_cached'].sum())} posts are already classified (cached). "
          f"{n_new} new posts need classification (the pilot in Section 5a estimates the requirement first).")
    posts_df.drop(columns=["source", "_cached"]).to_csv(RAW_POSTS_PATH, index=False)
    pd.DataFrame(collection_log, columns=["source", "platform", "status"]).to_csv(f"{OUT_DIR}/collection_log.csv", index=False)
    print(f"Saved {len(posts_df)} posts to {RAW_POSTS_PATH}")
    posts_df = posts_df.drop(columns=["source", "_cached"])


## 4.1 Expanded collection (v4): additional queries and sources, strict pre-classification filter

Widens the search for both platforms symmetrically (more YouTube, Hacker News, Bluesky, and Mastodon queries;
Lemmy as an additional source) and admits a post only if it passes `v4_gate`: English, names the platform,
contains AI vocabulary, and is not a news-bridge post. A YouTube preflight reports API errors explicitly.
Progress is saved in `expansion_state_v4.json`; already-collected posts are de-duplicated.

This expansion was undertaken because the previous round (v3) yielded a small HarmonyOS sample (106 AI-related
posts; 57 clusters, 20.3 effective). The reportability rule of Section 6 was not changed, and the same query policy,
filter, and verification were applied to both platforms.

In [ ]:
import collections
yt_err_log = collections.Counter()
RETRY_YOUTUBE = False

AI_VOCAB_STRICT = (r"\bai\b|a\.i\.|artificial intelligence|\bllm|chatbot|assistant|\bagents?\b|apple intelligence|writing tools?|"
                   r"genmoji|image playground|\bsiri\b|private cloud compute|harmony intelligence|celia|xiao ?yi|pangu|deepseek|"
                   r"copilot|gemini|\bgpt|summariz|on-?device model|visual intelligence")
AI_PREFILTER = re.compile(AI_VOCAB_STRICT, re.IGNORECASE)
HN_TITLE_TERMS = {
    "apple": re.compile(r"apple intelligence|apple ai|apple foundation|\bsiri\b|iphone|ipad|genmoji|image playground|"
                        r"private cloud compute|visual intelligence|writing tools|\bios ?(18|26)\b", re.IGNORECASE),
    "harmonyos": re.compile(r"harmony ?os|huawei|celia|xiao ?yi|hongmeng|pangu|openharmony", re.IGNORECASE)}

EXP_STATE_PATH  = f"{OUT_DIR}/expansion_state_v4.json"
V3_YT_STATE     = f"{OUT_DIR}/youtube_state_v3.json"
EXP_SINCE_ISO   = "2024-06-01T00:00:00Z"
EXP_SINCE_TS    = 1717200000
YT_UNIT_BUDGET_EXP          = 9000
YT_MAX_SEARCH_CALLS_PER_RUN = 45
YT_APPLE_MAX_NEW_VIDEOS     = 300
YT_CAP = 40
USE_MASTODON = True

EXP_YT_QUERIES = {
  "harmonyos": [
    ("HarmonyOS 6 Celia", True), ("Celia AI voice assistant Huawei", True), ("Xiaoyi assistant Huawei phone", True),
    ("HarmonyOS intelligent agents", True), ("HarmonyOS agent framework", True), ("Huawei AI features Mate 70", True),
    ("Huawei Pura 80 AI features", True), ("Huawei AI photo editing", True), ("Huawei AI call translation", True),
    ("Huawei DeepSeek Celia", True), ("HarmonyOS 5 AI features", True), ("HarmonyOS NEXT AI", True),
    ("HarmonyOS PC AI", True), ("Huawei MatePad AI features", True), ("Huawei Mate XT AI", True),
    ("Huawei Mate 80 AI", True), ("Huawei nova AI features", True), ("Huawei AI vs Apple Intelligence", True),
    ("Huawei AI vs Galaxy AI", True), ("Huawei AI zoom ultra clarity", True), ("Huawei AI retouch de-glare camera", True),
    ("Huawei Celia agent hands on", True), ("HarmonyOS 6 AI hands on", True), ("HarmonyOS AI assistant test", True),
    ("Huawei Xiaoyi voice assistant demo", True), ("Huawei AI features worth it", True), ("Huawei AI features disappointing", True),
    ("HarmonyOS 6 first look", False), ("HarmonyOS 5.1 review", False), ("Huawei Mate 80 review", False),
    ("HarmonyOS PC review", False), ("HarmonyOS vs iOS", False), ("HarmonyOS vs Android", False),
    ("Huawei Pura 80 Ultra review", False), ("Huawei Mate X6 review", False), ("HarmonyOS 6 beta", False),
  ],
  "apple": [
    ("Apple Intelligence ChatGPT integration", True), ("Visual Intelligence iPhone", True),
    ("Live Translation AirPods", True), ("Apple Intelligence iPhone 16 review", True),
    ("Apple Intelligence iPhone 17", True), ("Apple Intelligence BBC notification summaries", True),
    ("Apple Intelligence Siri delay", True), ("Apple Intelligence Mail summaries", True),
    ("Apple Intelligence is useless", True), ("Apple Intelligence months later", True),
    ("Apple Intelligence Mac", True), ("Apple Intelligence iPad", True),
    ("iOS 26 Apple Intelligence features", True), ("Apple Intelligence vs Pixel AI", True),
    ("Apple Intelligence vs Gemini", True), ("Apple Foundation Models on-device", True),
    ("new Siri iOS 26 test", True), ("Apple Intelligence honest review", True),
    ("iOS 26 long term review", False), ("iPhone 17 review AI", False),
  ],
}
EXP_TARGETED_PLATFORMS = ["harmonyos", "apple"]
EXP_TARGETED_TERMS = {"harmonyos": ["AI", "assistant", "Celia", "Xiaoyi", "agent", "DeepSeek"],
                      "apple": ["Apple Intelligence", "Siri", "Genmoji"]}
YT_APPLE_TARGETED_MAX_VIDEOS = 250

EXP_HN_QUERIES = {
  "harmonyos": [("Huawei Pangu", True), ("Huawei Celia", True), ("HarmonyOS 5", False), ("HarmonyOS 6", False),
                ("Huawei agent", True), ("Huawei DeepSeek", True), ("Huawei Mate AI", True), ("Huawei Pura AI", True),
                ("Huawei phone AI", True), ("OpenHarmony", False), ("HarmonyOS PC", False),
                ("Harmony Intelligence", True), ("Xiaoyi agent", True), ("Celia agent", True), ("HarmonyOS assistant", True),
                ("Huawei Mate 80", False), ("Huawei Pura 80", False), ("HarmonyOS AI agent", True)],
  "apple":     [("Apple Intelligence review", True), ("Apple Intelligence Siri", True), ("Visual Intelligence", True),
                ("Apple Foundation Models", True), ("Apple on-device model", True),
                ("Apple notification summaries BBC", True), ("Siri Gemini", True), ("Apple ChatGPT integration", True),
                ("Apple Intelligence disappointing", True), ("Genmoji", True), ("Image Playground", True),
                ("Live Translation AirPods", True)],
}
EXP_HN_STORY_QUERIES = {"harmonyos": ["HarmonyOS", "Huawei", "Huawei Pangu", "OpenHarmony", "Huawei Mate", "Huawei AI"],
                        "apple": ["Apple Intelligence", "Siri", "Apple Foundation Models", "Genmoji", "Visual Intelligence"]}
HN_STORY_MIN_COMMENTS = 5

EXP_BSKY_QUERIES = {
  "harmonyos": [("Huawei AI", True), ("Huawei Pura", False), ("Huawei Mate AI", True), ("HarmonyOS 5", False),
                ("HarmonyOS 6", False), ("HarmonyOS NEXT", False), ("Huawei assistant Celia", True), ("Xiaoyi", True),
                ("Huawei Pangu", True), ("Huawei DeepSeek", True), ("Celia Huawei", True), ("HarmonyOS agent", True),
                ("HarmonyOS assistant", True), ("Huawei Mate 80", False), ("Pura 80 AI", True), ("Harmony Intelligence", True),
                ("Huawei AI camera", True), ("HarmonyOS AI features", True)],
  "apple":     [("Apple Intelligence review", True), ("Apple Intelligence useless", True),
                ("Apple Intelligence summaries wrong", True), ("Visual Intelligence iPhone", True),
                ("Live Translation AirPods", True), ("Siri delay Apple Intelligence", True),
                ("iOS 26 Apple Intelligence", True), ("Apple Intelligence Mail", True),
                ("Genmoji", True), ("Image Playground", True), ("Apple Intelligence writing tools", True)],
}
EXP_MASTO_TAGS = {"harmonyos": ["harmonyos", "huawei", "openharmony", "hongmengos", "harmonyos6", "huaweimate80", "celiaai"],
                  "apple": ["appleintelligence", "genmoji", "siri", "applewritingtools", "visualintelligence"]}
EXP_MASTO_INSTANCES = ["mastodon.social", "mastodon.online", "fosstodon.org", "hachyderm.io", "mas.to", "infosec.exchange"]

USE_LEMMY = True
LEMMY_INSTANCES = ["lemmy.world", "lemmy.ml", "sh.itjust.works", "lemmy.zip", "programming.dev"]
LEMMY_QUERIES = {"harmonyos": ["HarmonyOS", "Huawei Celia", "Huawei Pangu", "HarmonyOS NEXT", "Huawei AI"],
                 "apple": ["Apple Intelligence", "Genmoji", "Siri Apple Intelligence", "Visual Intelligence", "Image Playground"]}
LEMMY_MAX_PAGES = 5

OWN_TERMS = {
    "apple": re.compile(r"apple intelligence|apple ai|\bapple\b|iphone|ipad|\bsiri\b|genmoji|image playground|writing tools?|"
                        r"visual intelligence|private cloud compute|\bios ?(18|26)\b|\bmacos\b", re.I),
    "harmonyos": re.compile(r"harmony ?os|huawei|hongmeng|openharmony|harmony intelligence", re.I)}
OTHER_TERMS = {
    "apple": re.compile(r"windows|copilot|galaxy|samsung|pixel|android|huawei|harmony", re.I),
    "harmonyos": re.compile(r"apple intelligence|\bapple\b|iphone|\bsiri\b|windows|copilot|galaxy|samsung|pixel|gemini", re.I)}
AI_STRICT = re.compile(r"\bai\b|a\.i\.|artificial intelligence|\bllm|chatbot|assistant|\bagents?\b|apple intelligence|writing tools?|"
                       r"genmoji|image playground|\bsiri\b|private cloud compute|harmony intelligence|celia|xiao ?yi|pangu|"
                       r"deepseek|copilot|gemini|\bgpt|summariz|summaris|on-?device model|visual intelligence|live translation|"
                       r"ai (zoom|photo|camera|retouch)|translat(e|ion) (call|feature)", re.I)
NEWS_BRIDGE_RE = re.compile(r"brid\.gy|gadgetbond|gizmodo|engadget|infosertecla|9to5|macrumors|theverge", re.I)
NEWS_PATTERN_RE = re.compile(r"^\s*\[\d+/\d+\]|\((?:sina|ithome|36kr|gizchina|huawei ?central|gsmarena)[^)]*\)", re.I)
gate_rejects = collections.Counter()

def v4_gate(p):
    text, ctx, plat, url = p["text"], p.get("context", "") or "", p["platform"], p["url"]
    both = f"{text} {ctx}"
    if len(text) < MIN_TEXT_LENGTH:
        return "short"
    if not mostly_english(text):
        return "non_english"
    if NEWS_BRIDGE_RE.search(url) or NEWS_PATTERN_RE.search(text):
        return "news_bridge"
    if not OWN_TERMS[plat].search(both):
        return "no_platform_term"
    if not AI_STRICT.search(both):
        return "no_ai_term"
    if OTHER_TERMS[plat].search(text) and not OWN_TERMS[plat].search(text):
        return "off_platform"
    if "youtube.com" not in url and not (OWN_TERMS[plat].search(text) or AI_STRICT.search(text)):
        return "no_topic_in_text"
    return None

def yt_preflight():
    global YOUTUBE_API_KEY
    YOUTUBE_API_KEY = (YOUTUBE_API_KEY or "").strip().strip("'\"")
    print(f"  YouTube key check: length {len(YOUTUBE_API_KEY)} (Google API keys are 39 characters), "
          f"starts with 'AIza': {YOUTUBE_API_KEY.startswith('AIza')}")
    tests = [("videos", {"part": "statistics", "id": "dQw4w9WgXcQ"}),
             ("commentThreads", {"part": "snippet", "videoId": "dQw4w9WgXcQ", "maxResults": 1, "searchTerms": "AI"}),
             ("search", {"part": "snippet", "q": "HarmonyOS 6 Celia", "type": "video", "maxResults": 1})]
    ok = True
    for ep, params in tests:
        try:
            r = requests.get(f"https://www.googleapis.com/youtube/v3/{ep}", params=dict(params, key=YOUTUBE_API_KEY), timeout=20)
            body = r.json() if r.headers.get("content-type", "").startswith("application/json") else {}
            reason = (body.get("error", {}).get("errors") or [{}])[0].get("reason") or ""
            emsg = body.get("error", {}).get("message", "")
            print(f"  YouTube preflight {ep}: HTTP {r.status_code} {reason} | {emsg[:140]}")
            if r.status_code != 200 and ep != "commentThreads":
                ok = False
            if r.status_code != 200 and ep == "commentThreads" and reason not in ("commentsDisabled", "videoNotFound"):
                ok = False
        except Exception as e:
            print(f"  YouTube preflight {ep}: {e}")
            ok = False
    if not ok:
        print("  HTTP 400 on every endpoint (even videos.list) means the KEY ITSELF is rejected, not the quota: re-copy it from "
              "Google Cloud Console > Credentials into the Colab secret (no quotes/spaces/newline), and confirm YouTube Data API v3 is "
              "enabled for that project.")
        print("  Hints: quotaExceeded -> wait for the midnight-Pacific reset; accessNotConfigured -> enable YouTube Data API v3 "
              "in the Google Cloud project; keyInvalid/forbidden -> check the key and its API/referrer restrictions.")
    return ok

if os.path.exists(EXP_STATE_PATH):
    xs = json.load(open(EXP_STATE_PATH))
    print(f"Resuming expansion: {len(xs['posts'])} new posts so far, sources done: {xs['done_sources']}")
else:
    xs = {"yt_searched": [], "yt_videos": {}, "yt_done": [], "yt_tdone": [], "posts": [], "done_sources": []}
def save_xs():
    json.dump(xs, open(EXP_STATE_PATH, "w"))

if RETRY_YOUTUBE and "youtube" in xs["done_sources"]:
    xs["done_sources"].remove("youtube"); xs["yt_searched"] = []; xs["yt_tdone"] = []
    print("RETRY_YOUTUBE: YouTube part reset (already-collected posts are de-duplicated, nothing is fetched twice).")
    if NO_NEW_SPEND:
        print("  WARNING: NO_NEW_SPEND=True -> new YouTube posts will be dropped in 3f. Use this only to DIAGNOSE the errors.")
_old = pd.read_csv(RAW_POSTS_PATH)
seen_exp = {(r.platform, r.url) for r in _old[["platform", "url"]].itertuples()}
seen_exp |= {(p["platform"], p["url"]) for p in xs["posts"]}
def add_post(p):
    if (p["platform"], p["url"]) in seen_exp:
        return False
    why = v4_gate(p)
    if why:
        gate_rejects[why] += 1
        return False
    seen_exp.add((p["platform"], p["url"]))
    xs["posts"].append(p)
    return True

def strip_html(s):
    return BeautifulSoup(html_lib.unescape(s or ""), "html.parser").get_text(" ", strip=True)

YT = "https://www.googleapis.com/youtube/v3"
class YTQuotaStop(Exception):
    pass
def yt_get(endpoint, params):
    params = dict(params, key=YOUTUBE_API_KEY)
    try:
        r = requests.get(f"{YT}/{endpoint}", params=params, timeout=20)
        time.sleep(REQUEST_DELAY)
    except Exception as e:
        yt_err_log[f"{endpoint}: network"] += 1
        return None, f"network: {e}"
    if r.status_code == 200:
        return r.json(), None
    try:
        reason = r.json()["error"]["errors"][0]["reason"]
    except Exception:
        reason = f"HTTP {r.status_code}"
    yt_err_log[f"{endpoint}: {reason}"] += 1
    if reason in ("quotaExceeded", "dailyLimitExceeded", "rateLimitExceeded"):
        raise YTQuotaStop(reason)
    return None, reason
def slug(q):
    return re.sub(r"\W+", "_", q.lower()).strip("_")

def yt_take(items, v, platform, tag):
    kept = 0
    for item in items:
        top = item["snippet"]["topLevelComment"]
        for c in [top] + item.get("replies", {}).get("comments", []):
            if kept >= YT_CAP:
                return kept
            text = c["snippet"].get("textOriginal", "").strip()
            url = f"https://www.youtube.com/watch?v={v['vid']}&lc={c['id']}"
            if (platform, url) in seen_exp or len(text) < MIN_TEXT_LENGTH or not mostly_english(text):
                continue
            if not passes_filter(text, platform, v["ai_specific"]):
                continue
            if add_post({"url": url, "platform": platform, "feature_tag": "yt_" + tag,
                         "username": c["snippet"].get("authorDisplayName", "unknown"), "text": text,
                         "context": f"Comment on YouTube video: {v['title']}", "cluster": "yt:" + v["vid"]}):
                kept += 1
    return kept

if "youtube" in xs["done_sources"]:
    print("YouTube expansion already complete (delete 'youtube' from done_sources in the state file to redo).")
elif not YOUTUBE_API_KEY:
    print("No YOUTUBE_API_KEY -- skipping YouTube expansion.")
    collection_log.append(("youtube_exp", "both", "no API key"))
elif not yt_preflight():
    print("YouTube skipped until the API problem above is fixed (nothing was marked done). Re-run this cell afterwards.")
else:
    known, done_v3 = {}, set()
    if os.path.exists(V3_YT_STATE):
        _s = json.load(open(V3_YT_STATE))
        known, done_v3 = _s["videos"], set(_s["done"])
    else:
        for r in _old[_old["cluster"].astype(str).str.startswith("yt:")].drop_duplicates("cluster").itertuples():
            vid = r.cluster[3:]
            known[f"{r.platform}|{vid}"] = {"platform": r.platform, "vid": vid, "ai_specific": False,
                                            "title": str(r.context).replace("Comment on YouTube video: ", ""),
                                            "queries": []}
        print("youtube_state_v3.json not found -- targeted pass will use only videos that already have posts.")
    units, search_calls, search_limit_hit = 0, 0, False
    tgt_variant, tgt_fail, targeted_broken = None, 0, False
    try:
        for platform in EXP_TARGETED_PLATFORMS:
            vids = [v for v in known.values() if v["platform"] == platform and not v.get("ai_specific")
                    and v.get("comment_count", 1) > 0]
            vids.sort(key=lambda v: -v.get("comment_count", 0))
            if platform == "apple":
                vids = vids[:YT_APPLE_TARGETED_MAX_VIDEOS]
            n_new = 0
            for v in vids:
                if targeted_broken:
                    break
                for term in EXP_TARGETED_TERMS[platform]:
                    key = f"{v['vid']}|{term}"
                    if key in xs["yt_tdone"]:
                        continue
                    if units >= YT_UNIT_BUDGET_EXP:
                        raise YTQuotaStop("local unit budget reached")
                    variants = [{"part": "snippet,replies", "maxResults": 100, "textFormat": "plainText", "order": "relevance"},
                                {"part": "snippet", "maxResults": 100, "textFormat": "plainText"}]
                    data, err = None, None
                    for vi in ([tgt_variant] if tgt_variant is not None else [0, 1]):
                        data, err = yt_get("commentThreads", dict(variants[vi], videoId=v["vid"], searchTerms=term))
                        units += 1
                        if data is not None:
                            tgt_variant = vi
                            break
                        if err in ("commentsDisabled", "videoNotFound"):
                            break
                    if data is not None:
                        tgt_fail = 0
                        n_new += yt_take(data.get("items", []), v, platform, "targeted_" + slug(term))
                        xs["yt_tdone"].append(key)
                    elif err in ("commentsDisabled", "videoNotFound"):
                        xs["yt_tdone"].append(key)
                    else:
                        tgt_fail += 1
                        if tgt_fail >= 8:
                            targeted_broken = True
                            print(f"  targeted searchTerms pass DISABLED: 8 failures in a row (last error: {err}). "
                                  f"The API rejects searchTerms for these requests; skipping to the search pass.")
                            break
                save_xs()
            print(f"  [{platform}] targeted searchTerms pass: +{n_new} posts from {len(vids)} general videos")

        for platform, queries in EXP_YT_QUERIES.items():
            for query, ai_specific in queries:
                if f"{platform}|{query}" in xs["yt_searched"]:
                    continue
                token, q_failed = None, False
                for _ in range(2):
                    if search_calls >= YT_MAX_SEARCH_CALLS_PER_RUN or units >= YT_UNIT_BUDGET_EXP:
                        search_limit_hit = True
                        q_failed = True
                        break
                    p = {"part": "snippet", "q": query, "type": "video", "maxResults": 50, "relevanceLanguage": "en",
                         "order": "relevance", "publishedAfter": EXP_SINCE_ISO}
                    if token:
                        p["pageToken"] = token
                    data, err = yt_get("search", p)
                    search_calls += 1
                    units += 100
                    if err or not data:
                        collection_log.append(("youtube_exp", platform, f"search '{query}': {err}"))
                        q_failed = True
                        break
                    for it in data.get("items", []):
                        vid = it.get("id", {}).get("videoId")
                        if not vid:
                            continue
                        k = f"{platform}|{vid}"
                        v = xs["yt_videos"].setdefault(k, {"platform": platform, "vid": vid, "queries": [], "ai_specific": False,
                                                           "title": html_lib.unescape(it["snippet"].get("title", ""))})
                        if query not in v["queries"]:
                            v["queries"].append(query)
                        v["ai_specific"] = v["ai_specific"] or ai_specific
                    token = data.get("nextPageToken")
                    if not token:
                        break
                if not q_failed:
                    xs["yt_searched"].append(f"{platform}|{query}")
                save_xs()
                if search_limit_hit:
                    break
        allv = {**known, **{k: {**known.get(k, {}), **v} for k, v in xs["yt_videos"].items()}}
        need = sorted({v["vid"] for v in allv.values() if "comment_count" not in v})
        for i in range(0, len(need), 50):
            chunk = need[i:i + 50]
            data, err = yt_get("videos", {"part": "statistics", "id": ",".join(chunk)})
            units += 1
            counts = {it["id"]: int(it.get("statistics", {}).get("commentCount", 0)) for it in (data or {}).get("items", [])}
            for v in allv.values():
                if v["vid"] in counts:
                    v["comment_count"] = counts[v["vid"]]
        for k, v in xs["yt_videos"].items():
            if "comment_count" in allv[k]:
                v["comment_count"] = allv[k]["comment_count"]
        save_xs()
        for platform in ["harmonyos", "apple"]:
            vids = [k for k, v in allv.items() if v["platform"] == platform and k not in done_v3 and k not in xs["yt_done"]
                    and v.get("comment_count", 1) > 0]
            vids.sort(key=lambda k: (not allv[k].get("ai_specific"), -allv[k].get("comment_count", 0)))
            if platform == "apple":
                vids = vids[:YT_APPLE_MAX_NEW_VIDEOS]
            got = 0
            for k in vids:
                v = allv[k]
                kept, token, vfail = 0, None, False
                for _ in range(3):
                    if units >= YT_UNIT_BUDGET_EXP:
                        raise YTQuotaStop("local unit budget reached")
                    p = {"part": "snippet,replies", "videoId": v["vid"], "maxResults": 100, "textFormat": "plainText",
                         "order": "relevance"}
                    if token:
                        p["pageToken"] = token
                    data, err = yt_get("commentThreads", p)
                    units += 1
                    if err or not data:
                        vfail = err not in ("commentsDisabled", "videoNotFound")
                        break
                    kept += yt_take(data.get("items", []), v, platform, slug((v.get("queries") or ["unknown"])[0]))
                    token = data.get("nextPageToken")
                    if kept >= YT_CAP or not token:
                        break
                got += kept
                if not vfail:
                    xs["yt_done"].append(k)
                save_xs()
            print(f"  [{platform}] new-video comments: +{got} from {len(vids)} videos")
        if search_limit_hit:
            print("YouTube: today's search/unit limit reached. Comments of the videos found so far were collected; re-run this cell "
                  "later (quota resets at midnight Pacific) to finish the remaining queries and videos.")
        else:
            xs["done_sources"].append("youtube")
            if yt_err_log:
                print("Note: some YouTube calls failed (reasons below); failed items were not marked done.")
    except YTQuotaStop as e:
        save_xs()
        print(f"YouTube stopped early ({e}). Progress saved -- re-run this cell after the quota resets (midnight Pacific).")
        collection_log.append(("youtube_exp", "both", f"stopped early: {e} (resumable)"))
    print(f"YouTube units used this run: {units} | search calls: {search_calls}")
    print("YouTube error reasons this run:", dict(yt_err_log) if yt_err_log else "none")

if "hn" not in xs["done_sources"]:
    n_kw = n_thr = 0
    for platform, queries in EXP_HN_QUERIES.items():
        for query, ai_specific in queries:
            for page in range(10):
                data, err = get_json("https://hn.algolia.com/api/v1/search", params={
                    "query": query, "tags": "comment", "hitsPerPage": 100, "page": page,
                    "numericFilters": f"created_at_i>{EXP_SINCE_TS}"})
                if err or not data:
                    collection_log.append(("hn_exp", platform, f"'{query}' p{page}: {err}"))
                    break
                for h in data.get("hits", []):
                    text = strip_html(h.get("comment_text"))
                    if len(text) < MIN_TEXT_LENGTH or not passes_filter(text, platform, ai_specific, require_platform_term=True):
                        continue
                    oid = h.get("objectID")
                    n_kw += add_post({"url": f"https://news.ycombinator.com/item?id={oid}", "platform": platform,
                                      "feature_tag": "hn_" + slug(query), "username": h.get("author", "unknown"), "text": text,
                                      "context": f"Hacker News comment on the story: {h.get('story_title') or ''}",
                                      "cluster": f"hn:{h.get('story_id') or oid}"})
                if page + 1 >= data.get("nbPages", 0):
                    break
    for platform, queries in EXP_HN_STORY_QUERIES.items():
        stories = {}
        for query in queries:
            for page in range(3):
                data, err = get_json("https://hn.algolia.com/api/v1/search", params={
                    "query": query, "tags": "story", "hitsPerPage": 100, "page": page,
                    "numericFilters": f"created_at_i>{EXP_SINCE_TS},num_comments>={HN_STORY_MIN_COMMENTS}"})
                if err or not data:
                    break
                for h in data.get("hits", []):
                    title = h.get("title") or ""
                    if HN_TITLE_TERMS[platform].search(title):
                        stories[h["objectID"]] = title
                if page + 1 >= data.get("nbPages", 0):
                    break
        for sid, title in stories.items():
            item, err = get_json(f"https://hn.algolia.com/api/v1/items/{sid}")
            if err or not item:
                continue
            stack, kept = list(reversed(item.get("children", []))), 0
            while stack and kept < YT_CAP:
                ch = stack.pop()
                stack.extend(reversed(ch.get("children", [])))
                text = strip_html(ch.get("text"))
                if len(text) < MIN_TEXT_LENGTH or not passes_filter(text, platform, False):
                    continue
                if add_post({"url": f"https://news.ycombinator.com/item?id={ch.get('id')}", "platform": platform,
                             "feature_tag": "hn_thread", "username": ch.get("author") or "unknown", "text": text,
                             "context": f"Hacker News comment on the story: {title}", "cluster": f"hn:{sid}"}):
                    kept += 1
                    n_thr += 1
        print(f"  [{platform}] HN threads scanned: {len(stories)}")
    xs["done_sources"].append("hn"); save_xs()
    print(f"Hacker News expansion: +{n_kw} keyword hits, +{n_thr} thread comments")

if "bluesky" not in xs["done_sources"]:
    if not (BLUESKY_HANDLE and BLUESKY_APP_PASSWORD):
        print("Skipping Bluesky expansion: no credentials.")
    else:
        try:
            r = requests.post("https://bsky.social/xrpc/com.atproto.server.createSession",
                              json={"identifier": BLUESKY_HANDLE.strip().lstrip("@").strip(),
                                    "password": BLUESKY_APP_PASSWORD.strip()}, timeout=20)
            r.raise_for_status()
            auth = {"Authorization": f"Bearer {r.json()['accessJwt']}"}
        except Exception as e:
            auth = None
            print(f"Bluesky login failed ({e}) -- skipping.")
        if auth:
            n = 0
            for platform, queries in EXP_BSKY_QUERIES.items():
                for query, ai_specific in queries:
                    cursor = None
                    for _ in range(10):
                        p = {"q": query, "limit": 100, "lang": "en", "since": EXP_SINCE_ISO}
                        if cursor:
                            p["cursor"] = cursor
                        data, err = get_json("https://bsky.social/xrpc/app.bsky.feed.searchPosts", params=p, headers=auth)
                        if err or not data:
                            collection_log.append(("bluesky_exp", platform, f"'{query}': {err}"))
                            break
                        for post in data.get("posts", []):
                            uri = post.get("uri", "")
                            text = ((post.get("record") or {}).get("text") or "").strip()
                            if len(text) < MIN_TEXT_LENGTH or not passes_filter(text, platform, ai_specific, require_platform_term=True):
                                continue
                            handle = post.get("author", {}).get("handle", "unknown")
                            n += add_post({"url": f"https://bsky.app/profile/{handle}/post/{uri.split('/')[-1]}",
                                           "platform": platform, "feature_tag": "bsky_" + slug(query), "username": handle,
                                           "text": text, "context": "Bluesky post", "cluster": "bsky:" + handle})
                        cursor = data.get("cursor")
                        if not cursor or not data.get("posts"):
                            break
            xs["done_sources"].append("bluesky"); save_xs()
            print(f"Bluesky expansion: +{n}")

if USE_MASTODON and "mastodon" not in xs["done_sources"]:
    n = 0
    for platform, tags in EXP_MASTO_TAGS.items():
        for inst in EXP_MASTO_INSTANCES:
            for tag in tags:
                max_id = None
                for _ in range(15):
                    p = {"limit": 40}
                    if max_id:
                        p["max_id"] = max_id
                    data, err = get_json(f"https://{inst}/api/v1/timelines/tag/{tag}", params=p)
                    if err or not data:
                        collection_log.append(("mastodon", platform, f"{inst}#{tag}: {err}"))
                        break
                    for st in data:
                        text = strip_html(st.get("content"))
                        if (st.get("language") not in (None, "en") or st.get("created_at", "") < EXP_SINCE_ISO[:10]
                                or len(text) < MIN_TEXT_LENGTH or not passes_filter(text, platform, False, require_platform_term=True)):
                            continue
                        acct = (st.get("account") or {}).get("acct", "unknown")
                        n += add_post({"url": st.get("url") or st.get("uri"), "platform": platform, "feature_tag": "masto_" + tag,
                                       "username": acct, "text": text, "context": "Mastodon post", "cluster": "masto:" + acct})
                    max_id = data[-1]["id"]
    xs["done_sources"].append("mastodon"); save_xs()
    print(f"Mastodon expansion: +{n}")

if USE_LEMMY and "lemmy" not in xs["done_sources"]:
    n_lem = 0
    for platform, queries in LEMMY_QUERIES.items():
        for inst in LEMMY_INSTANCES:
            for query in queries:
                for page in range(1, LEMMY_MAX_PAGES + 1):
                    data, err = get_json(f"https://{inst}/api/v3/search",
                                         params={"q": query, "type_": "Comments", "sort": "New", "limit": 50, "page": page})
                    if err or not data:
                        collection_log.append(("lemmy", platform, f"{inst} '{query}' p{page}: {err}"))
                        break
                    rows_ = data.get("comments", [])
                    if not rows_:
                        break
                    for it in rows_:
                        cm = it.get("comment", {}) or {}
                        post = it.get("post", {}) or {}
                        text = strip_html(cm.get("content"))
                        if (cm.get("published") or "") < EXP_SINCE_ISO[:10]:
                            continue
                        n_lem += add_post({"url": cm.get("ap_id") or f"https://{inst}/comment/{cm.get('id')}",
                                           "platform": platform, "feature_tag": "lemmy_" + slug(query),
                                           "username": (it.get("creator") or {}).get("name", "unknown"), "text": text,
                                           "context": f"Lemmy comment on the post: {post.get('name') or ''}",
                                           "cluster": f"lemmy:{inst}:{post.get('id')}"})
    xs["done_sources"].append("lemmy"); save_xs()
    print(f"Lemmy expansion: +{n_lem}")

print("\nCandidates rejected by the strict gate (before classification):", dict(gate_rejects))

print("\nNew raw posts this expansion (before caps/dedup against text):")
print(pd.DataFrame(xs["posts"]).groupby(["platform", "feature_tag"]).size().groupby("platform").sum().to_string()
      if xs["posts"] else "none")


## 4.2 Merge (v4)

De-duplicates, applies the per-cluster cap (keeping already-classified posts first when a cap forces a choice),
backs up the v4 file once, and prints how many posts are new and require classification.

In [ ]:
EXP_BACKUP_PATH = RAW_POSTS_PATH.replace(".csv", "_before_v4_expansion.csv")
cols = ["url", "platform", "feature_tag", "username", "text", "context", "cluster"]
old = pd.read_csv(RAW_POSTS_PATH)
old["context"] = old["context"].fillna("")
if not os.path.exists(EXP_BACKUP_PATH):
    old.to_csv(EXP_BACKUP_PATH, index=False)
    print("Backup written:", EXP_BACKUP_PATH)

new = pd.DataFrame(xs["posts"], columns=cols)
comb = pd.concat([old[cols], new], ignore_index=True)
comb = comb.drop_duplicates(subset=["platform", "url"]).drop_duplicates(subset=["platform", "text"]).reset_index(drop=True)
_cache = load_cache()
comb["_cached"] = [post_key(r) in _cache for _, r in comb.iterrows()]
if NO_NEW_SPEND:
    _left_out = int((~comb["_cached"]).sum())
    comb = comb[comb["_cached"]].copy()
    print(f"NO_NEW_SPEND: {_left_out} not-yet-classified posts left out of the dataset (nothing will be sent to Claude).")

def prefer_cached(g, n, seed=42):
    if len(g) <= n:
        return g
    c = g[g["_cached"]].sample(frac=1, random_state=seed)
    u = g[~g["_cached"]].sample(frac=1, random_state=seed)
    return pd.concat([c, u]).head(n)

comb = pd.concat([prefer_cached(g, MAX_POSTS_PER_CLUSTER) for _, g in comb.groupby(["platform", "cluster"])],
                 ignore_index=True)
comb["source"] = comb["url"].str.extract(r"https?://(?:www\.)?([^/]+)")[0]

was = set(zip(old["platform"], old["url"]))
comb["is_new_post"] = [(p, u) not in was for p, u in zip(comb["platform"], comb["url"])]
print("Before -> after expansion, by platform:")
print(pd.DataFrame({"before_posts": old.groupby("platform").size(),
                    "before_clusters": old.groupby("platform")["cluster"].nunique(),
                    "after_posts": comb.groupby("platform").size(),
                    "after_clusters": comb.groupby("platform")["cluster"].nunique()}).to_string())
print("\nNew posts by platform and source:")
print(comb[comb["is_new_post"]].groupby(["platform", "source"]).agg(new_posts=("url", "size"),
                                                                    new_clusters=("cluster", "nunique")).to_string())

n_new = int((~comb["_cached"]).sum())
est = n_new * cost_usd(670, 30, batch=True)
print(f"\nCLASSIFICATION PREVIEW: {int(comb['_cached'].sum())} posts already classified (cached). {n_new} new posts to classify, "
      f"~${est:.2f} via the Batch API (estimate; 5a re-measures it).")

posts_df = comb.drop(columns=["_cached", "source", "is_new_post"])
posts_df.to_csv(RAW_POSTS_PATH, index=False)
print(f"Saved {len(posts_df)} posts to {RAW_POSTS_PATH}")
print("Next: 5 (resume point) -> 5a (pilot) -> 5b (batch, budget-capped) -> 5c -> 5d (Claude precision pass) -> 5e (rules audit) -> 6.")


## 5. Classification with Claude

Only posts without a cached result are sent. Section 5.1 runs a pilot on a sample of new posts and projects the
requirement for the rest; Section 5.2 sends the remaining posts through the Message Batches API, only when
`PROCEED_WITH_FULL_EXTRACTION = True` and within `MAX_BUDGET_USD`, and resumes a submitted batch rather than
resubmitting it.

In [ ]:
try:
    posts_df
    print(f"posts_df in memory: {len(posts_df)} posts")
except NameError:
    posts_df = pd.read_csv(RAW_POSTS_PATH)
    posts_df["context"] = posts_df["context"].fillna("")
    print(f"Reloaded {len(posts_df)} saved posts")
DROP_UNCACHED_NOW = False
UNPROC_PATH = f"{OUT_DIR}/unprocessable_v4.json"
_unproc = json.load(open(UNPROC_PATH)) if os.path.exists(UNPROC_PATH) else {}
_c0 = load_cache()
_drop = [(post_key(r) in _unproc) or (DROP_UNCACHED_NOW and post_key(r) not in _c0) for _, r in posts_df.iterrows()]
if any(_drop):
    for _, r in posts_df[_drop].iterrows():
        print(f"  excluded [{r['platform']}] {str(r['text'])[:160]!r}")
    posts_df = posts_df[[not d for d in _drop]].reset_index(drop=True)
    print(f"Excluded {sum(_drop)} unprocessable post(s) (empty model response); disclose this in the paper.")
_c = load_cache()
_paid = sum(post_key(r) in _c for _, r in posts_df.iterrows())
print(posts_df.groupby("platform").agg(posts=("url", "size"), clusters=("cluster", "nunique")).to_string())
print(f"Already classified (cached): {_paid} | still to classify: {len(posts_df) - _paid}")


## 5.1 Pilot

## 5.2 Full classification (Message Batches API)

In [ ]:
PROCEED_WITH_FULL_EXTRACTION = not NO_NEW_SPEND
MAX_BUDGET_USD = 30.00
POLL_MINUTES = 30
BATCH_STATE_PATH = f"{OUT_DIR}/pending_batch.json"

UNPROC_PATH = f"{OUT_DIR}/unprocessable_v4.json"
def record_unprocessable(key):
    d = json.load(open(UNPROC_PATH)) if os.path.exists(UNPROC_PATH) else {}
    d[key] = d.get(key, 0) + 1
    json.dump(d, open(UNPROC_PATH, "w"))

def ingest_batch(state):
    counts = {"succeeded": 0, "parse_error": 0, "errored": 0, "canceled": 0, "expired": 0}
    in_sum = out_sum = 0
    for entry in client.messages.batches.results(state["batch_id"]):
        key = state["id_map"].get(entry.custom_id)
        rtype = entry.result.type
        if rtype != "succeeded":
            counts[rtype] = counts.get(rtype, 0) + 1
            continue
        msg = entry.result.message
        in_sum += msg.usage.input_tokens
        out_sum += msg.usage.output_tokens
        if check_response(msg):
            counts["truncated"] = counts.get("truncated", 0) + 1
            continue
        try:
            result = parse_json_response(response_text(msg))
        except Exception:
            counts["parse_error"] += 1
            if msg.usage.output_tokens == 0:
                record_unprocessable(key)
            continue
        append_cache(key, result, msg.usage.input_tokens, msg.usage.output_tokens, "batch")
        counts["succeeded"] += 1
    os.remove(BATCH_STATE_PATH)
    print("Batch results ingested:", counts)
    print(f"Batch API usage cost: ~${cost_usd(in_sum, out_sum, batch=True):.2f} "
          f"({in_sum:,} input / {out_sum:,} output tokens)")
    if sum(v for k, v in counts.items() if k != "succeeded"):
        print("Some requests did not succeed; they were NOT cached. Re-run this cell to resubmit only those.")

def poll_and_ingest(state):
    deadline = time.time() + POLL_MINUTES * 60
    while True:
        b = client.messages.batches.retrieve(state["batch_id"])
        rc = b.request_counts
        print(f"[{time.strftime('%H:%M:%S')}] batch {b.id}: {b.processing_status} | "
              f"processing={rc.processing} succeeded={rc.succeeded} errored={rc.errored}")
        if b.processing_status == "ended":
            ingest_batch(state)
            return True
        if time.time() > deadline:
            print(f"\nStill running on Anthropic's servers (can take up to 24h). Safe to close Colab.\n"
                  f"Re-run THIS cell later: it will resume batch {b.id} -- it will NOT resubmit or re-charge.")
            return False
        time.sleep(60)

if os.path.exists(BATCH_STATE_PATH):
    state = json.load(open(BATCH_STATE_PATH))
    print(f"Resuming existing batch {state['batch_id']} ({len(state['id_map'])} requests) -- not resubmitting.")
    poll_and_ingest(state)
else:
    cache = load_cache()
    pending = posts_df[[post_key(r) not in cache for _, r in posts_df.iterrows()]]
    print(f"Posts already processed (cached): {len(posts_df) - len(pending)} | pending: {len(pending)}")
    if len(pending) == 0:
        print("Nothing to send -- every post is already processed.")
    else:
        rep = pd.read_csv(f"{OUT_DIR}/pilot_report.csv") if os.path.exists(f"{OUT_DIR}/pilot_report.csv") else None
        if rep is None or rep.empty:
            print("Run the pilot (Section 5a) first; it provides the projection this gate relies on.")
        else:
            est = 0.0
            for platform, g in pending.groupby("platform"):
                r = rep[rep["platform"] == platform]
                if len(r):
                    est += len(g) * cost_usd(float(r["avg_in_tok"].iloc[0]), float(r["avg_out_tok"].iloc[0]), batch=True)
                else:
                    est += len(g) * cost_usd(rep["avg_in_tok"].max(), rep["avg_out_tok"].max(), batch=True)
            print(f"Projected cost for {len(pending)} pending posts (Batch API): ~${est:.2f} "
                  f"(budget cap ${MAX_BUDGET_USD:.2f})")
            if not PROCEED_WITH_FULL_EXTRACTION:
                print("\nNOT SENT. Review the pilot report above; if acceptable, set "
                      "PROCEED_WITH_FULL_EXTRACTION = True and re-run this cell.")
            elif est > MAX_BUDGET_USD:
                print("\nNOT SENT: projection exceeds MAX_BUDGET_USD. Raise the cap deliberately, "
                      "or lower TARGET_POSTS_PER_PLATFORM, to proceed.")
            else:
                id_map, requests_list = {}, []
                for _, row in pending.iterrows():
                    k = post_key(row)
                    cid = hashlib.sha1(k.encode()).hexdigest()[:40]
                    id_map[cid] = k
                    requests_list.append({"custom_id": cid, "params": {
                        "model": EXTRACTION_MODEL, "max_tokens": MAX_TOKENS, "thinking": THINKING,
                        "system": EXTRACTION_SYSTEM_PROMPT,
                        "messages": [{"role": "user", "content": build_user_msg(row)}]}})
                batch = client.messages.batches.create(requests=requests_list)
                state = {"batch_id": batch.id, "id_map": id_map, "submitted_at": time.time(),
                         "model": EXTRACTION_MODEL, "prompt_version": PROMPT_VERSION}
                json.dump(state, open(BATCH_STATE_PATH, "w"))
                print(f"Submitted batch {batch.id} with {len(requests_list)} requests. State saved to Drive.")
                poll_and_ingest(state)


In [ ]:
cache = load_cache()
rows, processed, ai_posts = [], 0, 0
for _, row in posts_df.iterrows():
    rec = cache.get(post_key(row))
    if not rec:
        continue
    processed += 1
    res = rec["result"]
    if res["ai_related"] == "Yes" and res["summaries"]:
        ai_posts += 1
        for s, sent in res["summaries"].items():
            rows.append({"platform": row["platform"], "feature_tag": row["feature_tag"],
                         "url": row["url"], "cluster": row["cluster"], "summary": s, "sentiment": sent})
extracted_df = pd.DataFrame(rows, columns=["platform", "feature_tag", "url", "cluster", "summary", "sentiment"])
print(f"Posts: {len(posts_df)} | processed: {processed} | AI-related: {ai_posts} | pairs: {len(extracted_df)}")
if processed < len(posts_df):
    print(f"WARNING: {len(posts_df) - processed} posts not processed yet -- results below are PARTIAL. "
          "Finish 5b before using them in the paper.")
if len(extracted_df):
    print(extracted_df.groupby(["platform", "sentiment"]).size().to_string())
extracted_df.to_csv(f"{OUT_DIR}/extracted_aspect_sentiment_v4_all.csv", index=False)
spent = sum(cost_usd(r["in_tok"], r["out_tok"], batch=(r.get("mode") == "batch")) for r in cache.values()
            if r["key"].startswith(PROMPT_VERSION))
print(f"Total spent on this prompt version so far: ~${spent:.2f}")
extracted_df_all = extracted_df.copy()


## 5.3 Verification pass

The extraction step accepts any post with some AI-related opinion. This pass applies one stricter question to every
AI-related post of both platforms (is it an opinion about the platform's own OS-level AI feature?) and stores a
category: AI-feature opinion, general OS, other platform, third-party app, LLM research or company news,
news/promotion, or other. It has its own cache and fingerprint and a budget cap (`MAX_VERIFY_BUDGET_USD`).
With `APPLY_VERIFIER = True`, Sections 6-8 use only posts verified as AI-feature opinions;
`INCLUDE_LLM_RESEARCH` optionally re-includes the LLM-research category without re-verification.

In [ ]:
import hashlib, json, os, time
from concurrent.futures import ThreadPoolExecutor

VERIFY_PLATFORMS = ["harmonyos", "apple"]
VERIFY_MODE = "batch"
PROCEED_WITH_VERIFICATION = not NO_NEW_SPEND
MAX_VERIFY_BUDGET_USD = 25.00
VERIFY_POLL_MINUTES = 30
INCLUDE_LLM_RESEARCH = False
APPLY_VERIFIER = True

VERIFY_SYSTEM_PROMPT = '''You are a strict relevance auditor for a study of operating-system AI features.
Platforms:
- apple: Apple Intelligence features (Writing Tools, Clean Up, Genmoji, Image Playground, notification/email summaries,
  Siri with Apple Intelligence, Visual Intelligence, Live Translation, ChatGPT integration).
- harmonyos: Huawei HarmonyOS system AI features (Celia/XiaoYi assistant and agents, Harmony Intelligence, AI photo/camera
  features such as AI zoom, retouching or de-glare, AI call translation, other AI built into the OS).

You receive a post, the context it was written in, and phrases already extracted from it. Decide whether the post genuinely
expresses an opinion, an issue, or a positive attribute about THAT platform's OS-level AI feature(s).

Answer "No" and pick the category when the post is mainly about:
- general_os: the OS in general (speed, battery, apps, ecosystem, UI, price, benchmarks, security) with no AI-feature opinion
- other_platform: another company's AI or OS (Windows/Copilot, Samsung/Galaxy AI, Google/Pixel/Gemini, Android, or the other platform of the study)
- third_party_app: an app's own AI (WPS, the ChatGPT app, Office, etc.) rather than the OS feature
- llm_research_or_company: foundation-model research, benchmarks, model releases, plagiarism or licensing controversy, chips, sanctions, geopolitics, corporate strategy
- news_or_promo: a headline, press release, advert or bare link with no user opinion
- other: anything else off-topic or unintelligible

Answer "Yes" (category ai_feature_opinion) only when the opinion is about the platform's own OS-level AI feature(s).
Return ONLY this JSON object, no other text:
{"relevant": "Yes" or "No", "category": "<ai_feature_opinion|general_os|other_platform|third_party_app|llm_research_or_company|news_or_promo|other>"}
'''
VERIFY_MODEL = EXTRACTION_MODEL
VERIFY_MAX_TOKENS = 200
VERIFY_THINKING = {"type": "disabled"}
VERIFY_VERSION = hashlib.sha1((VERIFY_SYSTEM_PROMPT + VERIFY_MODEL + json.dumps(VERIFY_THINKING, sort_keys=True)
                               + str(VERIFY_MAX_TOKENS)).encode()).hexdigest()[:10]
VERIFY_CATS = {"ai_feature_opinion", "general_os", "other_platform", "third_party_app", "llm_research_or_company",
               "news_or_promo", "other"}
VERIFY_CACHE_PATH = f"{OUT_DIR}/verify_cache_v4.jsonl"
VERIFY_STATE_PATH = f"{OUT_DIR}/pending_verify_batch.json"
print("Verifier fingerprint:", VERIFY_VERSION)

def vkey(platform, url):
    return f"{VERIFY_VERSION}|{platform}|{url}"

def load_vcache():
    c = {}
    if os.path.exists(VERIFY_CACHE_PATH):
        for line in open(VERIFY_CACHE_PATH, encoding="utf-8"):
            try:
                rec = json.loads(line)
                c[rec["key"]] = rec
            except Exception:
                continue
    return c

def append_vcache(key, result, in_tok, out_tok, mode):
    rec = {"key": key, "result": result, "in_tok": in_tok, "out_tok": out_tok, "mode": mode}
    with open(VERIFY_CACHE_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(rec) + "\n")
    return rec

def parse_verify(raw):
    text = re.sub(r"```(?:json)?", "", raw)
    start = text.find("{")
    if start == -1:
        raise ValueError("no JSON object")
    obj, _ = json.JSONDecoder().raw_decode(text[start:])
    rel = str(obj.get("relevant", "")).strip().lower() == "yes"
    cat = str(obj.get("category", "")).strip().lower()
    cat = cat if cat in VERIFY_CATS else "other"
    if rel:
        cat = "ai_feature_opinion"
    elif cat == "ai_feature_opinion":
        cat = "other"
    return {"relevant": rel, "category": cat}

def build_verify_msg(row):
    return (f"Platform: {row['platform']}. Context: {str(row['context'])[:300]}. "
            f"Phrases already extracted: {str(row['phrases'])[:300]}.\n\nPost:\n{str(row['text'])[:2000]}")

def verify_targets():
    base = extracted_df_all
    ph = (base[base["platform"].isin(VERIFY_PLATFORMS)].groupby(["platform", "url"])["summary"]
          .agg(lambda s: "; ".join(s)).reset_index().rename(columns={"summary": "phrases"}))
    ph = ph.merge(posts_df[["platform", "url", "text", "context"]].drop_duplicates(["platform", "url"]),
                  on=["platform", "url"], how="left")
    ph["text"] = ph["text"].fillna(""); ph["context"] = ph["context"].fillna("")
    ph["vkey"] = [vkey(p, u) for p, u in zip(ph["platform"], ph["url"])]
    return ph

def _vcost(in_tok, out_tok, batch):
    return cost_usd(in_tok, out_tok, batch)

def verify_sync(row):
    try:
        resp = client.messages.create(model=VERIFY_MODEL, max_tokens=VERIFY_MAX_TOKENS, thinking=VERIFY_THINKING,
                                      system=VERIFY_SYSTEM_PROMPT,
                                      messages=[{"role": "user", "content": build_verify_msg(row)}])
        if getattr(resp, "stop_reason", None) == "max_tokens":
            return row["vkey"], None, resp.usage.input_tokens, resp.usage.output_tokens, "truncated"
        return row["vkey"], parse_verify(response_text(resp)), resp.usage.input_tokens, resp.usage.output_tokens, None
    except Exception as e:
        return row["vkey"], None, 0, 0, str(e)[:200]

def ingest_verify_batch(state):
    counts, in_sum, out_sum = {"succeeded": 0, "parse_error": 0, "errored": 0, "canceled": 0, "expired": 0}, 0, 0
    for entry in client.messages.batches.results(state["batch_id"]):
        key = state["id_map"].get(entry.custom_id)
        if entry.result.type != "succeeded":
            counts[entry.result.type] = counts.get(entry.result.type, 0) + 1
            continue
        msg = entry.result.message
        in_sum += msg.usage.input_tokens; out_sum += msg.usage.output_tokens
        if getattr(msg, "stop_reason", None) == "max_tokens":
            counts["parse_error"] += 1
            continue
        try:
            res = parse_verify(response_text(msg))
        except Exception:
            counts["parse_error"] += 1
            continue
        append_vcache(key, res, msg.usage.input_tokens, msg.usage.output_tokens, "batch")
        counts["succeeded"] += 1
    os.remove(VERIFY_STATE_PATH)
    print("Verifier batch ingested:", counts, f"| cost ~${_vcost(in_sum, out_sum, True):.2f}")
    if sum(v for k, v in counts.items() if k != "succeeded"):
        print("Some requests failed and were NOT cached; re-run this cell to resubmit only those.")

def poll_verify(state):
    deadline = time.time() + VERIFY_POLL_MINUTES * 60
    while True:
        b = client.messages.batches.retrieve(state["batch_id"])
        rc = b.request_counts
        print(f"[{time.strftime('%H:%M:%S')}] verify batch {b.id}: {b.processing_status} | processing={rc.processing} "
              f"succeeded={rc.succeeded} errored={rc.errored}")
        if b.processing_status == "ended":
            ingest_verify_batch(state)
            return True
        if time.time() > deadline:
            print("Still running on Anthropic's servers; re-run THIS cell later -- it resumes the same batch (no re-charge).")
            return False
        time.sleep(60)

vt = verify_targets()
vc = load_vcache()
pending = vt[~vt["vkey"].isin(vc)]
print(f"AI-related posts to verify: {len(vt)} | already verified (cached): {len(vt) - len(pending)} | pending: {len(pending)}")

if os.path.exists(VERIFY_STATE_PATH):
    st = json.load(open(VERIFY_STATE_PATH))
    print(f"Resuming verifier batch {st['batch_id']} ({len(st['id_map'])} requests) -- not resubmitting.")
    poll_verify(st)
elif len(pending):
    est_in = sum((len(VERIFY_SYSTEM_PROMPT) + len(build_verify_msg(r))) / 3.6 for _, r in pending.iterrows())
    est_out = 25 * len(pending)
    est = _vcost(est_in, est_out, VERIFY_MODE == "batch")
    print(f"Estimated cost ({VERIFY_MODE}, {VERIFY_MODEL}): ~${est:.2f} for {len(pending)} posts (cap ${MAX_VERIFY_BUDGET_USD:.2f})")
    if not PROCEED_WITH_VERIFICATION:
        print("NOT SENT. Set PROCEED_WITH_VERIFICATION = True to run.")
    elif est > MAX_VERIFY_BUDGET_USD:
        print("NOT SENT: estimate exceeds MAX_VERIFY_BUDGET_USD. Raise the cap deliberately or restrict VERIFY_PLATFORMS.")
    elif VERIFY_MODE == "sync":
        errs, spent_in, spent_out = [], 0, 0
        with ThreadPoolExecutor(max_workers=8) as ex:
            for key, res, it, ot, err in ex.map(verify_sync, [r for _, r in pending.iterrows()]):
                spent_in += it; spent_out += ot
                if err:
                    errs.append(err); continue
                append_vcache(key, res, it, ot, "sync")
        print(f"Sync verification done: {len(pending) - len(errs)} ok, {len(errs)} errors | cost ~${_vcost(spent_in, spent_out, False):.2f}")
        for e in errs[:3]:
            print("  error:", e)
    else:
        id_map, reqs = {}, []
        for _, r in pending.iterrows():
            cid = hashlib.sha1(r["vkey"].encode()).hexdigest()[:40]
            id_map[cid] = r["vkey"]
            reqs.append({"custom_id": cid, "params": {"model": VERIFY_MODEL, "max_tokens": VERIFY_MAX_TOKENS,
                                                       "thinking": VERIFY_THINKING, "system": VERIFY_SYSTEM_PROMPT,
                                                       "messages": [{"role": "user", "content": build_verify_msg(r)}]}})
        batch = client.messages.batches.create(requests=reqs)
        st = {"batch_id": batch.id, "id_map": id_map, "submitted_at": time.time(), "version": VERIFY_VERSION}
        json.dump(st, open(VERIFY_STATE_PATH, "w"))
        print(f"Submitted verifier batch {batch.id} ({len(reqs)} requests). State saved to Drive.")
        poll_verify(st)

vc = load_vcache()
rows_ = []
for _, r in vt.iterrows():
    rec = vc.get(r["vkey"])
    if rec:
        rows_.append({"platform": r["platform"], "url": r["url"], **rec["result"]})
verif_df = pd.DataFrame(rows_, columns=["platform", "url", "relevant", "category"])
missing = len(vt) - len(verif_df)
ok_cats = {"ai_feature_opinion"} | ({"llm_research_or_company"} if INCLUDE_LLM_RESEARCH else set())
verif_df["keep"] = verif_df["category"].isin(ok_cats)
verif_df.to_csv(f"{OUT_DIR}/verifier_verdicts_v4.csv", index=False)

if len(verif_df):
    print("\nVerifier verdicts by platform and category (share of AI-related posts):")
    print(pd.crosstab(verif_df["platform"], verif_df["category"], margins=True).to_string())
    print("\nKept as relevant (%):", (verif_df.groupby("platform")["keep"].mean() * 100).round(1).to_dict())
if missing:
    print(f"\nWARNING: {missing} posts are still unverified -- extracted_df is left UNCHANGED (all posts). "
          f"Finish verification (re-run this cell) before using results in the paper.")
elif APPLY_VERIFIER:
    kept = extracted_df_all.merge(verif_df[["platform", "url", "keep"]], on=["platform", "url"], how="left")
    kept["keep"] = kept["keep"].fillna(True)
    extracted_df = kept[kept["keep"]].drop(columns="keep").reset_index(drop=True)
    extracted_df.to_csv(f"{OUT_DIR}/extracted_aspect_sentiment_v4_verified.csv", index=False)
    print(f"\nAPPLY_VERIFIER=True: extracted_df = verified-relevant posts only "
          f"({len(extracted_df)} of {len(extracted_df_all)} pairs). Sections 6-8 use this set. "
          f"The unfiltered set stays in extracted_df_all.")
else:
    print("\nAPPLY_VERIFIER=False: extracted_df unchanged (all AI-related posts).")
vspent = sum(_vcost(r["in_tok"], r["out_tok"], r.get("mode") == "batch") for r in vc.values() if r["key"].startswith(VERIFY_VERSION))
print(f"Total spent on the verifier so far: ~${vspent:.2f}")


## 5.4 Rule-based audit and relevance sensitivity (no API calls)

Flags doubtful posts with transparent rules (platform terms, language, news-style threads, LLM controversies,
third-party apps) and recomputes the Section 6 statistics under five scenarios: A (as classified), B (strict rules),
C (rules excluding LLM controversies), V (verification pass), and V+B. Writes `relevance_audit_v4.csv`, the input
for the human validation.

In [ ]:
import re, os
import numpy as np
import pandas as pd

APPLY_STRICT_FILTER = False
STRICT_SCENARIO = "V+B"
AUDIT_APPLE_SAMPLE = 100
_N_BOOT = 3000
MIN_CLUSTERS_TO_REPORT = globals().get("MIN_CLUSTERS_TO_REPORT", 30)
MAX_TOP1_SHARE_TO_REPORT = globals().get("MAX_TOP1_SHARE_TO_REPORT", 0.20)

OWN = {
    "apple": re.compile(r"apple intelligence|apple ai|\bapple\b|iphone|ipad|\bsiri\b|genmoji|image playground|writing tools?|"
                        r"visual intelligence|private cloud compute|\bios ?(18|26)\b|\bmacos\b", re.I),
    "harmonyos": re.compile(r"harmony ?os|huawei|hongmeng|openharmony|harmony intelligence", re.I)}
OTHER = {
    "apple": re.compile(r"windows|copilot|galaxy|samsung|pixel|android|huawei|harmony", re.I),
    "harmonyos": re.compile(r"apple intelligence|\bapple\b|iphone|\bsiri\b|windows|copilot|galaxy|samsung|pixel|gemini", re.I)}
GENERIC_OS = re.compile(r"benchmark|performance|latency|blockchain|registration|network needed|polished|battery|"
                        r"update access|stable|workflow|\bads?\b|price|sanction|ecosystem|app support|smooth|fast\b", re.I)
AI_FEAT = re.compile(r"\bai\b|assistant|agent|celia|xiaoyi|siri|genmoji|summar|translat|voice|generat|llm|model|deepseek|"
                     r"pangu|intelligen|photo|zoom|clarity|retouch|gesture|writing|playground|clean ?up|chatgpt|gpt|"
                     r"copilot|recognition|transcri|proofread|search", re.I)
THIRD_PARTY = re.compile(r"\bwps\b|office ai|microsoft 365|google docs", re.I)
LLM_CONTROVERSY = re.compile(r"pangu|plagiar|allegation|shell model|cloned|openpangu|model release|imo ", re.I)
NEWS_BRIDGE = re.compile(r"brid\.gy|gadgetbond|gizmodo|engadget|infosertecla|9to5|macrumors|theverge", re.I)
NEWS_PATTERN = re.compile(r"^\s*\[\d+/\d+\]|\((?:sina|ithome|36kr|gizchina|huawei ?central|gsmarena)[^)]*\)", re.I)

pt = (extracted_df_all.groupby(["platform", "url", "cluster"])
      .agg(summaries=("summary", lambda s: "; ".join(s)), sents=("sentiment", lambda s: ",".join(s))).reset_index())
pt = pt.merge(posts_df[["platform", "url", "text", "context", "feature_tag", "username"]].drop_duplicates(["platform", "url"]),
              on=["platform", "url"], how="left")
pt["text"] = pt["text"].fillna(""); pt["context"] = pt["context"].fillna("")

def _flags(r):
    p, txt, ctx = r["platform"], r["text"], r["context"]
    both = f"{txt} {ctx}"
    return pd.Series({
        "no_platform_term": not OWN[p].search(both),
        "off_platform": bool(OTHER[p].search(txt)) and not OWN[p].search(txt) and not OWN[p].search(r["summaries"]),
        "non_english": not mostly_english(txt),
        "news_bridge": bool(NEWS_BRIDGE.search(r["url"])) or bool(NEWS_PATTERN.search(txt)),
        "third_party_app": bool(THIRD_PARTY.search(f"{txt} {r['summaries']}")),
        "llm_controversy": bool(LLM_CONTROVERSY.search(f"{txt} {r['summaries']}")),
    })
pt = pd.concat([pt, pt.apply(_flags, axis=1)], axis=1)
POST_DROP = ["no_platform_term", "off_platform", "non_english", "news_bridge", "third_party_app"]
pt["drop_B"] = pt[POST_DROP].any(axis=1)
pt["drop_C"] = pt["drop_B"] | pt["llm_controversy"]

ed = extracted_df_all.merge(pt[["platform", "url", "drop_B", "drop_C"]], on=["platform", "url"], how="left")
HAVE_V = "verif_df" in globals() and len(verif_df) > 0
if HAVE_V:
    ed = ed.merge(verif_df[["platform", "url", "keep", "category"]].rename(columns={"keep": "keep_v", "category": "v_category"}),
                  on=["platform", "url"], how="left")
    ed["keep_v"] = ed["keep_v"].fillna(True).astype(bool)
    pt = pt.merge(verif_df[["platform", "url", "category"]].rename(columns={"category": "v_category"}), on=["platform", "url"], how="left")
else:
    ed["keep_v"] = True; ed["v_category"] = ""
    pt["v_category"] = ""
ed["generic_phrase"] = ed["summary"].apply(lambda s: bool(GENERIC_OS.search(s)) and not AI_FEAT.search(s))

print("Flag counts among AI-related posts (a post can have several flags):")
print(pt.groupby("platform")[POST_DROP + ["llm_controversy", "drop_B", "drop_C"]].sum().assign(
    ai_posts=pt.groupby("platform").size()).to_string())
print("\nGeneric-OS phrases (no AI vocabulary), by platform:",
      ed.groupby("platform")["generic_phrase"].agg(["sum", "size"]).rename(columns={"sum": "flagged", "size": "pairs"}).to_dict("index"))

SCEN_NAMES = ["A (as classified)", "B (strict rules)", "C (rules + no LLM-controversy)"] + \
             (["V (verifier)", "V+B (verifier + rules)"] if HAVE_V else [])
def scenario(df, name):
    if name.startswith("A"):
        return df
    if name.startswith("V+B"):
        return df[df["keep_v"] & (~df["drop_B"].astype(bool)) & (~df["generic_phrase"])]
    if name.startswith("V"):
        return df[df["keep_v"]]
    col = "drop_B" if name.startswith("B") else "drop_C"
    return df[(~df[col].astype(bool)) & (~df["generic_phrase"])]

def _cboot(pol, n=_N_BOOT, seed=0):
    groups = [g["pos"].values for _, g in pol.groupby("cluster")]
    if len(groups) < 2:
        return (np.nan, np.nan)
    rng = np.random.default_rng(seed)
    b = [np.concatenate([groups[i] for i in rng.integers(0, len(groups), len(groups))]).mean() for _ in range(n)]
    return tuple(np.percentile(b, [2.5, 97.5]) * 100)

rows = []
for platform in ["harmonyos", "apple"]:
    for name in SCEN_NAMES:
        s = scenario(ed[ed["platform"] == platform], name)
        posts = s.drop_duplicates("url")
        per = posts.groupby("cluster").size().sort_values(ascending=False)
        pol = s[s["sentiment"].isin(["positive", "negative"])].copy()
        pol["pos"] = (pol["sentiment"] == "positive").astype(int)
        lo, hi = _cboot(pol) if len(pol) else (np.nan, np.nan)
        top1 = per.iloc[0] / per.sum() if len(per) else np.nan
        ok = len(per) >= MIN_CLUSTERS_TO_REPORT and top1 <= MAX_TOP1_SHARE_TO_REPORT
        rows.append({"platform": platform, "scenario": name, "ai_posts": len(posts), "clusters": len(per),
                     "kish_eff_clusters": round(per.sum() ** 2 / (per ** 2).sum(), 1) if len(per) else np.nan,
                     "top1_pct": round(100 * top1, 1), "pairs": len(s),
                     "positive_share_pct": round(100 * pol["pos"].mean(), 1) if len(pol) else np.nan,
                     "ci_low": round(lo, 1), "ci_high": round(hi, 1), "reportable": ok})
sens = pd.DataFrame(rows)
print("\n=== SENSITIVITY TO THE RELEVANCE RULES (no API calls) ===")
print(sens.to_string(index=False))
sens.to_csv(f"{OUT_DIR}/relevance_sensitivity_v4.csv", index=False)

ap = pt[pt["platform"] == "apple"]
audit = pd.concat([pt[pt["platform"] == "harmonyos"], ap.sample(min(len(ap), AUDIT_APPLE_SAMPLE), random_state=1)])
audit = audit.assign(text=audit["text"].str.slice(0, 500))[
    ["platform", "url", "cluster", "context", "text", "summaries", "sents", "no_platform_term", "off_platform", "non_english",
     "news_bridge", "third_party_app", "llm_controversy", "drop_B", "v_category"]]
audit["human_relevant_Y_N"] = ""; audit["human_sentiment"] = ""
_AUDIT_PATH = f"{OUT_DIR}/relevance_audit_v4.csv"
if os.path.exists(_AUDIT_PATH):
    _old_a = pd.read_csv(_AUDIT_PATH)
    for _col in ["human_relevant_Y_N", "human_sentiment"]:
        _old_a[_col] = _old_a[_col].fillna("").astype(str)
    _lab_old = _old_a[_old_a["human_relevant_Y_N"].str.strip() != ""].drop_duplicates("url")
    if len(_lab_old):
        for _col in ["human_relevant_Y_N", "human_sentiment"]:
            audit[_col] = audit["url"].map(_lab_old.set_index("url")[_col]).fillna("")
        _gone = _lab_old[~_lab_old["url"].isin(audit["url"])]
        audit = pd.concat([audit, _gone[audit.columns.intersection(_gone.columns)]], ignore_index=True)
        print(f"Kept {len(_lab_old)} existing human labels in the audit file ({len(_gone)} on rows outside the new sample).")
audit.to_csv(_AUDIT_PATH, index=False)
print(f"\nSaved audit file with {len(audit)} posts (all HarmonyOS AI-related + {min(len(ap), AUDIT_APPLE_SAMPLE)} Apple): "
      f"{OUT_DIR}/relevance_audit_v4.csv")

print("\nFlagged HarmonyOS posts (first 15) -- eyeball these to check that the rules are not over-dropping:")
for _, r in pt[(pt["platform"] == "harmonyos") & pt["drop_C"]].head(15).iterrows():
    why = [c for c in POST_DROP + ["llm_controversy"] if r[c]]
    print(f"- [{','.join(why)}] {r['text'][:140].replace(chr(10), ' ')!r} -> {r['summaries'][:80]}")

if APPLY_STRICT_FILTER:
    _map = {"B": "B (strict rules)", "C": "C (rules + no LLM-controversy)", "V": "V (verifier)", "V+B": "V+B (verifier + rules)"}
    name = _map[STRICT_SCENARIO]
    if name not in SCEN_NAMES:
        raise ValueError(f"Scenario {STRICT_SCENARIO} needs the verifier results (run 5d first).")
    extracted_df = scenario(ed, name)[["platform", "feature_tag", "url", "cluster", "summary", "sentiment"]].reset_index(drop=True)
    extracted_df.to_csv(f"{OUT_DIR}/extracted_aspect_sentiment_v4_{STRICT_SCENARIO.replace('+', '')}.csv", index=False)
    print(f"\nAPPLY_STRICT_FILTER=True: extracted_df now = scenario {STRICT_SCENARIO} ({len(extracted_df)} pairs). "
          f"Re-run Sections 6-8 to use it.")
else:
    print("\nextracted_df is UNCHANGED by this cell (APPLY_STRICT_FILTER=False). Set it to True and re-run to feed Sections 6-8 another scenario.")


## 5.5 Scope sensitivity: camera/photo AI versus other AI features (no API calls)

Recomputes the Section 6 statistic with and without camera- and photo-related pairs, to test whether the Apple
versus HarmonyOS difference depends on whether phone-camera AI is counted as an OS AI feature.

In [ ]:
import re
import numpy as np
import pandas as pd

def _cluster_bootstrap_ci(pol, n=5000, seed=0):
    groups = [g["pos"].values for _, g in pol.groupby("cluster")]
    if len(groups) < 2:
        return (float("nan"), float("nan"))
    rng = np.random.default_rng(seed)
    b = [np.concatenate([groups[i] for i in rng.integers(0, len(groups), len(groups))]).mean() for _ in range(n)]
    return tuple(np.percentile(b, [2.5, 97.5]) * 100)

CAMERA_RE = re.compile(r"camera|photo|portrait|zoom|pose|selfie|lens|snapshot|picture|night ?mode|low ?light|"
                       r"xmage|shot|clarity|blur|oversharpen|exposure|image quality", re.IGNORECASE)
NOT_CAMERA_RE = re.compile(r"generat|playground|genmoji|emoji", re.IGNORECASE)

def is_camera(summary):
    return bool(CAMERA_RE.search(summary)) and not NOT_CAMERA_RE.search(summary)

rows = []
for platform in ["apple", "harmonyos"]:
    sub = extracted_df[extracted_df["platform"] == platform].copy()
    if sub.empty:
        continue
    sub["camera"] = sub["summary"].astype(str).map(is_camera)
    for label, part in [("all AI-feature pairs (Section 6)", sub),
                        ("excluding camera/photo AI", sub[~sub["camera"]]),
                        ("camera/photo AI only", sub[sub["camera"]])]:
        pol = part[part["sentiment"].isin(["positive", "negative"])].copy()
        pol["pos"] = (pol["sentiment"] == "positive").astype(int)
        n_clusters = part["cluster"].nunique()
        if len(pol) < 20 or n_clusters < 2:
            rows.append({"platform": platform, "scope": label, "pairs": len(part), "clusters": n_clusters,
                         "positive_share_pct": None, "ci_low": None, "ci_high": None})
            continue
        lo, hi = _cluster_bootstrap_ci(pol)
        rows.append({"platform": platform, "scope": label, "pairs": len(part), "clusters": n_clusters,
                     "positive_share_pct": round(100 * pol["pos"].mean(), 1),
                     "ci_low": round(lo, 1), "ci_high": round(hi, 1)})

scope_report = pd.DataFrame(rows)
print(scope_report.to_string(index=False))
share_cam = {p: round(100 * extracted_df[extracted_df.platform == p]["summary"].astype(str).map(is_camera).mean(), 1)
             for p in ["apple", "harmonyos"] if (extracted_df.platform == p).any()}
print(f"\nShare of pairs that are camera/photo AI: {share_cam}")
print("Interpretation: if HarmonyOS's positive share stays above Apple's once camera/photo AI is excluded, the\n"
      "platform difference does not depend on that scope choice; if it falls to Apple's level, report it per scope.")
scope_report.to_csv(f"{OUT_DIR}/scope_sensitivity_camera_v4.csv", index=False)


## 5.6 Verification pass versus a single human label column (no API calls)

Computes precision, recall, and Cohen's kappa of the verification pass against the `human_relevant_Y_N` column
of `relevance_audit_v4.csv`. Superseded by the two-rater validation in Section 5.7, which was used for the paper.

In [ ]:
AUDIT_PATH = f"{OUT_DIR}/relevance_audit_v4.csv"
_a = pd.read_csv(AUDIT_PATH)
_a["human"] = _a["human_relevant_Y_N"].fillna("").astype(str).str.strip().str.upper().str[:1].map({"Y": 1, "N": 0})
_lab = _a.dropna(subset=["human"]).copy()
if _lab.empty:
    print(f"No human labels yet. Open {AUDIT_PATH}, fill human_relevant_Y_N (Y/N), save, and re-run this cell.")
else:
    print(f"Labeled rows: {len(_lab)} of {len(_a)} ({int(_lab['human'].sum())} Y, {int((1 - _lab['human']).sum())} N)."
          + ("  FEW LABELS: treat the statistics below as indicative only." if len(_lab) < 60 else ""))
    _yes = {"ai_feature_opinion"} | ({"llm_research_or_company"} if globals().get("INCLUDE_LLM_RESEARCH", False) else set())
    _lab["verifier"] = _lab["v_category"].isin(_yes).astype(int)
    _rows = []
    for plat, g in _lab.groupby("platform"):
        tp = int(((g["verifier"] == 1) & (g["human"] == 1)).sum()); fp = int(((g["verifier"] == 1) & (g["human"] == 0)).sum())
        fn = int(((g["verifier"] == 0) & (g["human"] == 1)).sum()); tn = int(((g["verifier"] == 0) & (g["human"] == 0)).sum())
        n = len(g); po = (tp + tn) / n
        pe = ((tp + fp) * (tp + fn) + (fn + tn) * (fp + tn)) / (n * n)
        _rows.append({"platform": plat, "labeled": n, "TP": tp, "FP": fp, "FN": fn, "TN": tn,
                      "precision": round(tp / (tp + fp), 3) if tp + fp else np.nan,
                      "recall": round(tp / (tp + fn), 3) if tp + fn else np.nan,
                      "accuracy": round(po, 3), "cohen_kappa": round((po - pe) / (1 - pe), 3) if pe < 1 else np.nan})
    print("Verifier vs human relevance labels (human = reference):")
    print(pd.DataFrame(_rows).to_string(index=False))
    _dis = _lab[_lab["verifier"] != _lab["human"]]
    print(f"\nDisagreements: {len(_dis)}. First 12 (read these; they show which rule in the verifier prompt to adjust):")
    for _, r in _dis.head(12).iterrows():
        print(f"- [{r['platform']}] verifier={r['v_category']} human={'Y' if r['human'] == 1 else 'N'} | {str(r['text'])[:140]!r}")
    print("\nReport precision/recall/kappa per platform in the paper. Low recall on HarmonyOS means the verifier is too strict there:")
    print("re-include categories by changing INCLUDE_LLM_RESEARCH, or relax the verification prompt (requires re-verification).")


## 5.7 Human validation with two independent raters (no API calls)

Each author labeled a blind workbook (`usability_labels_Alhawarat.xlsx`, `usability_labels_Nijem.xlsx`) containing
the 628 HarmonyOS first-pass AI-related posts and a random sample of 100 Apple posts, without any model output.
This cell computes inter-rater agreement and, once both workbooks are complete, writes
`usability_reconciliation.xlsx` listing every disagreement. After the disagreements were resolved by discussion
(`FINAL_relevant`, `FINAL_sentiment`), it computes the pipeline's precision, recall, and kappa against the
consensus labels. The labeling rulebook is reproduced in the folder README.

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score

RATER_FILES = {"Alhawarat": f"{OUT_DIR}/usability_labels_Alhawarat.xlsx", "Nijem": f"{OUT_DIR}/usability_labels_Nijem.xlsx"}
AUDIT_PATH = f"{OUT_DIR}/relevance_audit_v4.csv"
RECON_PATH = f"{OUT_DIR}/usability_reconciliation.xlsx"

def _rel(x):
    x = "" if pd.isna(x) else str(x).strip().upper()
    return {"Y": "Y", "N": "N"}.get(x[:1], np.nan)

def _sent(x):
    x = "" if pd.isna(x) else str(x).strip().lower()
    return x if x in {"positive", "negative", "neutral", "mixed"} else np.nan

def _model_sent(s):
    parts = [p.strip() for p in str(s).split(",") if p.strip()]
    pos, neg = parts.count("positive"), parts.count("negative")
    return "mixed" if (pos and neg) else "positive" if pos else "negative" if neg else "neutral"

def _kappa(a, b):
    a, b = list(a), list(b)
    return round(cohen_kappa_score(a, b), 3) if len(set(a) | set(b)) > 1 else np.nan

def load_raters():
    lab, complete = {}, True
    for name, path in RATER_FILES.items():
        d = pd.read_excel(path, sheet_name="Labels").rename(
            columns={"Item ID": "item_id", "Relevant? (Y/N)": "rel", "Sentiment (only if Y)": "sent"})
        d["rel"] = d["rel"].map(_rel); d["sent"] = d["sent"].map(_sent)
        done = int(d["rel"].notna().sum())
        print(f"{name}: {done} of {len(d)} posts labeled" + ("" if done == len(d) else "  (INCOMPLETE)"))
        complete = complete and done == len(d)
        lab[name] = d.set_index("item_id")[["rel", "sent"]]
    return lab, complete

def interrater(both, A, B):
    rows = []
    for plat, g in [("all", both)] + list(both.groupby("platform")):
        yy = g[(g.A_rel == "Y") & (g.B_rel == "Y")].dropna(subset=["A_sent", "B_sent"])
        rows.append({"platform": plat, "posts": len(g),
                     "relevance_agreement_pct": round(100 * (g.A_rel == g.B_rel).mean(), 1),
                     "relevance_kappa": _kappa(g.A_rel, g.B_rel), "both_Y": len(yy),
                     "sentiment_agreement_pct": round(100 * (yy.A_sent == yy.B_sent).mean(), 1) if len(yy) else np.nan,
                     "sentiment_kappa": _kappa(yy.A_sent, yy.B_sent) if len(yy) else np.nan})
    irr = pd.DataFrame(rows)
    print(f"\n=== Inter-rater agreement ({A} vs {B}), {len(both)} posts labeled by both ===")
    print(irr.to_string(index=False))
    irr.to_csv(f"{OUT_DIR}/usability_interrater_v4.csv", index=False)
    return irr

def disagreements(both):
    return both[(both.A_rel != both.B_rel) |
                ((both.A_rel == "Y") & (both.B_rel == "Y") & (both.A_sent != both.B_sent))]

def validate(both, dis, rec, irr):
    rec_rel = rec["FINAL_relevant"].map(_rel); rec_sent = rec["FINAL_sentiment"].map(_sent)
    unresolved = [i for i in dis.index if pd.isna(rec_rel.get(i, np.nan))]
    if unresolved:
        print(f"\n{len(unresolved)} of {len(dis)} disagreements still have no FINAL_relevant -- fill them, save, re-run.")
        return
    c = both.copy()
    rel_agree = c.A_rel == c.B_rel
    sent_agree = rel_agree & ((c.A_rel == "N") | (c.A_sent == c.B_sent))
    c["H_rel"] = [c.at[i, "A_rel"] if rel_agree[i] else rec_rel.get(i) for i in c.index]
    c["H_sent"] = [c.at[i, "A_sent"] if sent_agree[i] else rec_sent.get(i, np.nan) for i in c.index]
    c.loc[c.H_rel == "N", "H_sent"] = np.nan
    rows = []
    for plat, g in [("all", c)] + list(c.groupby("platform")):
        tp = int(((g.model_rel == "Y") & (g.H_rel == "Y")).sum()); fp = int(((g.model_rel == "Y") & (g.H_rel == "N")).sum())
        fn = int(((g.model_rel == "N") & (g.H_rel == "Y")).sum()); tn = int(((g.model_rel == "N") & (g.H_rel == "N")).sum())
        yy = g[(g.model_rel == "Y") & (g.H_rel == "Y")].dropna(subset=["H_sent"])
        rows.append({"platform": plat, "posts": len(g), "TP": tp, "FP": fp, "FN": fn, "TN": tn,
                     "precision": round(tp / (tp + fp), 3) if tp + fp else np.nan,
                     "recall": round(tp / (tp + fn), 3) if tp + fn else np.nan,
                     "relevance_kappa": _kappa(g.model_rel, g.H_rel), "sentiment_posts": len(yy),
                     "sentiment_agreement_pct": round(100 * (yy.model_sent == yy.H_sent).mean(), 1) if len(yy) else np.nan,
                     "sentiment_kappa": _kappa(yy.model_sent, yy.H_sent) if len(yy) else np.nan})
    val = pd.DataFrame(rows)
    print(f"\n=== Model vs human consensus (consensus = reference), {len(c)} posts ===")
    print(val.to_string(index=False))
    val.to_csv(f"{OUT_DIR}/usability_validation_v4.csv", index=False)
    c[["platform", "url", "A_rel", "A_sent", "B_rel", "B_sent", "H_rel", "H_sent", "model_rel", "model_sent",
       "v_category"]].to_csv(f"{OUT_DIR}/usability_consensus_labels_v4.csv")
    r, v = irr.iloc[0], val.iloc[0]
    print(f"\nFor the paper: two authors independently labeled {len(c)} posts (relevance kappa = {r.relevance_kappa}; "
          f"sentiment kappa = {r.sentiment_kappa}). Against their reconciled labels, the pipeline reached precision "
          f"{v.precision} and recall {v.recall} for relevance, and {v.sentiment_agreement_pct}% sentiment agreement "
          f"(kappa = {v.sentiment_kappa}).")

audit = pd.read_csv(AUDIT_PATH)
audit["item_id"] = [f"R{i+1:03d}" for i in range(len(audit))]
audit["model_rel"] = np.where(audit["v_category"] == "ai_feature_opinion", "Y", "N")
audit["model_sent"] = audit["sents"].map(_model_sent)
missing = [p for p in RATER_FILES.values() if not os.path.exists(p)]
if missing:
    print("Waiting for these files in OUT_DIR:", missing)
else:
    lab, complete = load_raters()
    A, B = list(lab)
    m = audit.set_index("item_id").join(lab[A].add_prefix("A_")).join(lab[B].add_prefix("B_"))
    both = m.dropna(subset=["A_rel", "B_rel"])
    irr = interrater(both, A, B)
    if not complete:
        print("\nBoth raters must finish ALL posts before the disagreement list is created (agreement above is provisional).")
    else:
        dis = disagreements(both)
        if os.path.exists(RECON_PATH):
            rec = pd.read_excel(RECON_PATH).set_index("item_id")
            validate(both, dis, rec, irr)
        else:
            out = dis[["platform", "context", "text", "url", "A_rel", "A_sent", "B_rel", "B_sent"]].copy()
            out.columns = ["platform", "context", "text", "url", f"{A}_relevant", f"{A}_sentiment",
                           f"{B}_relevant", f"{B}_sentiment"]
            out["FINAL_relevant"] = ""; out["FINAL_sentiment"] = ""
            out.reset_index().to_excel(RECON_PATH, index=False)
            print(f"\nWrote {len(out)} disagreements to {RECON_PATH}. Discuss them together, fill FINAL_relevant (Y/N) and "
                  f"FINAL_sentiment (only if Y), save, and run this cell again.")


## 6. Positive share with cluster-robust statistics

Cluster-bootstrap 95% confidence interval (5,000 resamples of whole clusters), Kish effective number of clusters,
concentration (top-1 and top-5 cluster shares), a sensitivity analysis capping each cluster at 10 posts, and the
reportability rule: at least 30 clusters, and no single cluster contributing more than 20% of AI-related posts.

In [ ]:
MIN_CLUSTERS_TO_REPORT = 30
MAX_TOP1_SHARE_TO_REPORT = 0.20
SENSITIVITY_CAP = 10
N_BOOT = 5000

def cluster_bootstrap_ci(pol, n=N_BOOT, seed=0):
    groups = [g["pos"].values for _, g in pol.groupby("cluster")]
    if len(groups) < 2:
        return (float("nan"), float("nan"))
    rng = np.random.default_rng(seed)
    b = [np.concatenate([groups[i] for i in rng.integers(0, len(groups), len(groups))]).mean() for _ in range(n)]
    return tuple(np.percentile(b, [2.5, 97.5]) * 100)

report = []
for platform in ["apple", "harmonyos"]:
    sub = extracted_df[extracted_df["platform"] == platform]
    if sub.empty:
        print(f"=== {platform}: no AI-related posts ===\n")
        report.append({"platform": platform, "reportable": False, "reason": "no AI-related posts"})
        continue
    posts = sub.drop_duplicates("url")
    per = posts.groupby("cluster").size().sort_values(ascending=False)
    n_eff = per.sum() ** 2 / (per ** 2).sum()
    top1, top5 = per.iloc[0] / per.sum(), per.head(5).sum() / per.sum()
    pol = sub[sub["sentiment"].isin(["positive", "negative"])].copy()
    pol["pos"] = (pol["sentiment"] == "positive").astype(int)
    share = 100 * pol["pos"].mean() if len(pol) else float("nan")
    lo, hi = cluster_bootstrap_ci(pol) if len(pol) else (float("nan"), float("nan"))
    keep = pd.concat([g.sample(min(len(g), SENSITIVITY_CAP), random_state=0) for _, g in posts.groupby("cluster")])["url"]
    sens = 100 * pol[pol["url"].isin(keep)]["pos"].mean() if len(pol) else float("nan")
    reasons = []
    if per.size < MIN_CLUSTERS_TO_REPORT:
        reasons.append(f"only {per.size} clusters (< {MIN_CLUSTERS_TO_REPORT})")
    if top1 > MAX_TOP1_SHARE_TO_REPORT:
        reasons.append(f"top cluster supplies {100*top1:.0f}% (> {100*MAX_TOP1_SHARE_TO_REPORT:.0f}%)")
    ok = not reasons
    dist = (sub["sentiment"].value_counts(normalize=True) * 100).round(1).to_dict()
    print(f"=== {platform} ===")
    print(f"AI-related posts: {len(posts)} from {per.size} clusters | effective clusters (Kish): {n_eff:.1f} | "
          f"top-1 share: {100*top1:.1f}% | top-5 share: {100*top5:.1f}%")
    print(f"Pairs: {len(sub)} | sentiment distribution (%): {dist}")
    print(f"Positive share (pos/(pos+neg)): {share:.1f}%  cluster-bootstrap 95% CI [{lo:.1f}, {hi:.1f}]  "
          f"| sensitivity (<= {SENSITIVITY_CAP}/cluster): {sens:.1f}%")
    print(f"REPORTABLE: {'YES' if ok else 'NO -- ' + '; '.join(reasons)}\n")
    report.append({"platform": platform, "ai_posts": len(posts), "clusters": per.size, "effective_clusters": round(n_eff, 1),
                   "top1_share_pct": round(100 * top1, 1), "top5_share_pct": round(100 * top5, 1), "pairs": len(sub),
                   "positive_share_pct": round(share, 1), "ci_low": round(lo, 1), "ci_high": round(hi, 1),
                   "sensitivity_pct": round(sens, 1), "reportable": ok, "reason": "; ".join(reasons)})
robustness_report = pd.DataFrame(report)
robustness_report.to_csv(f"{OUT_DIR}/robustness_report_v4.csv", index=False)
print(f"Saved {OUT_DIR}/robustness_report_v4.csv")


## 7. Clustering of positive and negative aspect phrases, per platform

Same method as `Copilot_Usability_Clustering.ipynb`: sentence-transformer embeddings (all-mpnet-base-v2), a k-means
sweep, and average-linkage merging of redundant clusters on centroid cosine distance.

In [ ]:
model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2', device=device)

def run_kmeans_sweep(embeddings, k_range, random_state=42):
    rows = []
    for k in k_range:
        if k >= len(embeddings):
            break
        km = KMeans(n_clusters=k, random_state=random_state, n_init=10)
        labels = km.fit_predict(embeddings)
        sil = silhouette_score(embeddings, labels)
        db = davies_bouldin_score(embeddings, labels)
        ch = calinski_harabasz_score(embeddings, labels)
        rows.append({'k': k, 'silhouette': sil, 'davies_bouldin': db,
                      'calinski_harabasz': ch, 'inertia': km.inertia_})
        print(f"k={k:2d}  silhouette={sil:.4f}  davies_bouldin={db:.4f}")
    return pd.DataFrame(rows)

def compute_centroids(embeddings, labels, k):
    return np.array([embeddings[labels == c].mean(axis=0) for c in range(k)])

def agglomerative_merge(sim_matrix, distance_threshold, method='average'):
    n = sim_matrix.shape[0]
    if n < 2:
        return [[0]] if n == 1 else [], None
    dist_matrix = 1 - sim_matrix
    np.fill_diagonal(dist_matrix, 0)
    dist_matrix = (dist_matrix + dist_matrix.T) / 2
    condensed = squareform(dist_matrix, checks=False)
    Z = linkage(condensed, method=method)
    labels = fcluster(Z, t=distance_threshold, criterion='distance')
    groups = {}
    for i, lab in enumerate(labels):
        groups.setdefault(lab, []).append(i)
    return list(groups.values()), Z

def summarize_clusters(phrases, labels, k, n_samples=10, n_top_terms=8):
    summary = []
    for c in range(k):
        idx = np.where(labels == c)[0]
        cluster_phrases = [phrases[i] for i in idx]
        size = len(cluster_phrases)
        try:
            vec = CountVectorizer(ngram_range=(1,2), stop_words='english', max_features=50)
            X = vec.fit_transform(cluster_phrases)
            term_counts = np.asarray(X.sum(axis=0)).flatten()
            terms = vec.get_feature_names_out()
            top_terms = [t for t,_ in sorted(zip(terms, term_counts), key=lambda x: -x[1])[:n_top_terms]]
        except ValueError:
            top_terms = []
        samples = list(np.random.RandomState(42).choice(cluster_phrases, min(n_samples, size), replace=False))
        summary.append({'cluster': c, 'size': size, 'pct_of_total': round(100*size/len(phrases), 2),
                         'top_terms': ', '.join(top_terms), 'sample_phrases': samples})
    return pd.DataFrame(summary).sort_values('size', ascending=False).reset_index(drop=True)

def summarize_merged(phrases, labels, merge_groups, n_samples=10, n_top_terms=8):
    summary = []
    for meta_id, group in enumerate(merge_groups):
        idx = np.concatenate([np.where(labels == c)[0] for c in group])
        cluster_phrases = [phrases[i] for i in idx]
        size = len(cluster_phrases)
        try:
            vec = CountVectorizer(ngram_range=(1,2), stop_words='english', max_features=50)
            X = vec.fit_transform(cluster_phrases)
            term_counts = np.asarray(X.sum(axis=0)).flatten()
            terms = vec.get_feature_names_out()
            top_terms = [t for t,_ in sorted(zip(terms, term_counts), key=lambda x: -x[1])[:n_top_terms]]
        except ValueError:
            top_terms = []
        samples = list(np.random.RandomState(42).choice(cluster_phrases, min(n_samples, size), replace=False))
        summary.append({'meta_cluster': meta_id, 'original_clusters': group, 'size': size,
                         'pct_of_total': round(100*size/len(phrases), 2),
                         'top_terms': ', '.join(top_terms), 'sample_phrases': samples})
    return pd.DataFrame(summary).sort_values('size', ascending=False).reset_index(drop=True)

DISTANCE_THRESHOLD = 0.45
LINKAGE_METHOD = 'average'

def cluster_platform(platform_name, extracted_df, distance_threshold=DISTANCE_THRESHOLD):
    sub = extracted_df[extracted_df['platform'] == platform_name].copy()
    sub['summary_clean'] = sub['summary'].str.strip().str.lower()
    results = {}
    for sentiment in ['positive', 'negative']:
        phrases = sub.loc[sub['sentiment']==sentiment, 'summary_clean'].tolist()
        if len(phrases) < 4:
            print(f"{platform_name}/{sentiment}: only {len(phrases)} phrases -- too few to cluster meaningfully.")
            results[sentiment] = None
            continue
        print(f"\n=== Clustering {platform_name}/{sentiment} ({len(phrases)} phrases) ===")
        embeddings = model.encode(phrases, show_progress_bar=True)
        max_k = max(2, min(30, len(phrases) // 10, len(phrases) - 1))
        sweep = run_kmeans_sweep(embeddings, range(2, max_k + 1))
        if len(sweep) == 0:
            results[sentiment] = None
            continue
        best_k = int(sweep.loc[sweep['silhouette'].idxmax(), 'k'])
        print(f"Silhouette-best k for {platform_name}/{sentiment}: {best_k}")
        km = KMeans(n_clusters=best_k, random_state=42, n_init=10)
        labels = km.fit_predict(embeddings)
        raw_summary = summarize_clusters(phrases, labels, best_k)

        centroids = compute_centroids(embeddings, labels, best_k)
        sim_matrix = cosine_similarity(centroids)
        merge_groups, Z = agglomerative_merge(sim_matrix, distance_threshold, LINKAGE_METHOD)
        merged_summary = summarize_merged(phrases, labels, merge_groups)
        print(f"  {best_k} raw clusters -> {len(merge_groups)} merged meta-themes "
              f"(distance_threshold={distance_threshold})")

        results[sentiment] = {
            'sweep': sweep, 'best_k': best_k, 'raw_summary': raw_summary,
            'sim_matrix': sim_matrix, 'merge_groups': merge_groups, 'Z': Z,
            'merged_summary': merged_summary, 'phrases': phrases, 'labels': labels,
        }
    return results

apple_results = cluster_platform('apple', extracted_df)


In [ ]:
harmonyos_results = cluster_platform('harmonyos', extracted_df)


## 7.1 Merge diagnostics

Centroid-similarity heatmaps and merge dendrograms for each platform and sentiment, with the distance cut
(`DISTANCE_THRESHOLD`) marked.

In [ ]:
def plot_diagnostics(results, platform_name):
    sentiments_with_data = [s for s in ['positive','negative'] if results.get(s) is not None]
    if not sentiments_with_data:
        print(f"{platform_name}: no clustered data to visualize.")
        return
    fig, axes = plt.subplots(1, len(sentiments_with_data), figsize=(7*len(sentiments_with_data), 6))
    if len(sentiments_with_data) == 1:
        axes = [axes]
    for ax, sentiment in zip(axes, sentiments_with_data):
        r = results[sentiment]
        im = ax.imshow(r['sim_matrix'], cmap='RdYlBu_r', vmin=0, vmax=1)
        ax.set_title(f'{platform_name}/{sentiment} centroid similarity (k={r["best_k"]})')
        plt.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/{platform_name}_centroid_similarity.png", dpi=300)
    plt.show()

    fig, axes = plt.subplots(1, len(sentiments_with_data), figsize=(7*len(sentiments_with_data), 5))
    if len(sentiments_with_data) == 1:
        axes = [axes]
    for ax, sentiment in zip(axes, sentiments_with_data):
        r = results[sentiment]
        if r['Z'] is not None and len(r['Z']) > 0:
            dendrogram(r['Z'], ax=ax, color_threshold=DISTANCE_THRESHOLD)
            ax.axhline(y=DISTANCE_THRESHOLD, color='red', linestyle='--')
        ax.set_title(f'{platform_name}/{sentiment} dendrogram')
    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/{platform_name}_dendrogram.png", dpi=300)
    plt.show()

plot_diagnostics(apple_results, 'apple')
plot_diagnostics(harmonyos_results, 'harmonyos')


## 8. Theme export

Prints representative phrases per merged theme and exports the theme table. Theme names in the paper were
assigned by the authors from these phrases.

In [ ]:
def print_clusters(results, platform_name):
    for sentiment in ['positive', 'negative']:
        r = results.get(sentiment)
        if r is None:
            continue
        print(f"\n{'='*80}\n{platform_name.upper()} -- {sentiment.upper()} MERGED META-THEMES "
              f"({r['best_k']} raw -> {len(r['merge_groups'])} merged)\n{'='*80}")
        for _, row in r['merged_summary'].iterrows():
            tag = f" [merged from raw clusters {row['original_clusters']}]" if len(row['original_clusters']) > 1 else ""
            print(f"\nMeta-cluster {row['meta_cluster']}{tag}  |  size={row['size']} ({row['pct_of_total']}%)")
            print(f"  Top terms: {row['top_terms']}")
            print(f"  Samples: {row['sample_phrases']}")

print_clusters(apple_results, 'apple')
print_clusters(harmonyos_results, 'harmonyos')


In [ ]:
export_rows = []
for platform_name, results in [('apple', apple_results), ('harmonyos', harmonyos_results)]:
    for sentiment in ['positive', 'negative']:
        r = results.get(sentiment)
        if r is None:
            continue
        for _, row in r['merged_summary'].iterrows():
            export_rows.append({
                'platform': platform_name, 'sentiment': sentiment, 'meta_cluster': row['meta_cluster'],
                'theme_name': row['top_terms'],
                'size': row['size'], 'pct_of_total': row['pct_of_total'],
                'merged_from_raw_clusters': ','.join(map(str, row['original_clusters'])),
                'sample_phrases': '; '.join(row['sample_phrases'][:5])
            })

theme_table = pd.DataFrame(export_rows)
theme_table.to_csv(f"{OUT_DIR}/apple_harmonyos_theme_clusters_v4.csv", index=False)
print(f"Saved theme table to {OUT_DIR}/apple_harmonyos_theme_clusters_v4.csv")
theme_table


In [ ]:
summary_stats = {}
for platform_name, results in [('apple', apple_results), ('harmonyos', harmonyos_results)]:
    sub = extracted_df[extracted_df['platform']==platform_name]
    platform_stats = {
        'total_pairs': int(len(sub)),
        'unique_source_posts': int(sub['url'].nunique()) if len(sub) else 0,
        'positive_pct': round(float((sub['sentiment']=='positive').mean()*100), 2) if len(sub) else None,
        'negative_pct': round(float((sub['sentiment']=='negative').mean()*100), 2) if len(sub) else None,
    }
    for sentiment in ['positive', 'negative']:
        r = results.get(sentiment)
        if r is not None:
            platform_stats[f'{sentiment}_raw_k'] = r['best_k']
            platform_stats[f'{sentiment}_merged_themes'] = len(r['merge_groups'])
    summary_stats[platform_name] = platform_stats

summary_stats['_merge_config'] = {'distance_threshold': DISTANCE_THRESHOLD, 'linkage_method': LINKAGE_METHOD}

with open(f"{OUT_DIR}/apple_harmonyos_summary_stats_v4.json", "w") as f:
    json.dump(summary_stats, f, indent=2)

print(json.dumps(summary_stats, indent=2))
print(f"\nAll outputs saved to: {OUT_DIR}")


## Notes on reporting

- **Classifier:** Claude Sonnet 5 (extended thinking disabled), one post per request, a single prompt for all
  Apple/HarmonyOS posts. The Windows Copilot corpus was classified by the pipeline of Chhetri et al. (2025).
- **Two-stage relevance:** extraction (Section 5) followed by the verification pass (Section 5.3); rule-based
  scenarios (Section 5.4) serve as sensitivity analyses.
- **Independence:** posts are clustered by video, thread, author, or app; each cluster is capped at 40;
  uncertainty is a cluster-bootstrap confidence interval.
- **Reportability rule:** a platform's sentiment is reported only with at least 30 clusters and no cluster above
  20% of AI-related posts.
- **Data-dependent expansion:** the v4 collection was undertaken because the v3 round yielded a small HarmonyOS
  sample (106 AI-related posts, 20.3 effective clusters); queries, filter, and verification were applied to both
  platforms alike, and the reportability rule was not changed.
- **Cluster capping:** when a cap forces a choice within a cluster, already-classified posts are kept first;
  within a video or thread this selects among comments the API returned in relevance order.
- **Human validation (Section 5.7):** the final HarmonyOS estimate uses the two authors' consensus labels directly;
  the final Apple estimate uses verified posts detected as English. These final estimates are computed in
  `Reproduce_Usability_Results.ipynb`.